# TDBRAIN MDD vs HC — frozen CBraMod (Mumtaz와 같은 설정)

**시작 전:** 메뉴 `런타임 → 런타임 유형 변경 → GPU` 선택 후, 위에서부터 한 셀씩 실행하세요.

- **데이터:** TDBRAIN V3.1 (Brainclinics), 안정 상태 EO·EC 각 약 2분, 26채널, 500 Hz, `.bdf`
- **과제:** 질병 유무 분류 (HC = 0, MDD = 1), 피험자 단위 층화 5-fold (seed 42)
- **대상:** `indication`이 HEALTHY 또는 MDD 단독, 18세 이상, 피험자당 첫 세션
- **코호트:** matched(성별 같고 나이 차 3세 이내 1:1, 주 결과) · full(보조)

| 실험 | 내용 |
|---|---|
| 1. EEG-only | matched · full 5-fold |
| 2. 조건별 단독 평가·잡음 점검 | EO만 / EC만, 그룹별 100 μV 초과 구간 비율 |
| 3. 임상정보 조합 | 맥락 변수 조합, 스태킹, 중첩 선택, EEG 비중 |
| 4. 정보 교체 | EEG는 그대로, 임상정보만 본인 / 무작위 / 같은 그룹 / 반대 그룹 것으로 교체 |
| 5. 통합 스크립트 재실행 | PD-EEG·AD-EEG와 같은 형식(`tdbrain_fusion.py`)으로 실험 1·3·4 + 관련성·EEG 근거 보존 결합 |

**설정:** 19채널(T3/T4/T5/T6 → T7/T8/P7/P8 이름 대응) · 재참조 없음 · 0.3–75 Hz · 50 Hz notch · 200 Hz · 5 s 비중첩 · μV/100 · frozen CBraMod(원본 repo + 원본 사전학습 가중치, 3,800차원) · StandardScaler + L2 로지스틱 회귀(C = 0.01)

> **데이터는 저장소에 포함되지 않습니다.** TDBRAIN은 사용 동의 후 Brainclinics에서 직접 받아야 하며, zip 비밀번호는 실행할 때 입력합니다.

## 0. 설치

In [ ]:
!pip install -q braindecode mne openpyxl
!apt -qq install -y aria2 p7zip-full > /dev/null 2>&1
import torch; print('GPU:', torch.cuda.is_available())

## 1. Google Drive 연결 + 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK  = '/content/drive/MyDrive/tdbrain_hcmdd_project'   # 결과·데이터 백업 위치
LOCAL = '/content/tdbrain_hcmdd'                          # 빠른 읽기용 로컬 복사본
XLSX  = f'{WORK}/TDBRAIN_participants_V3.1.xlsx'
OUT   = f'{WORK}/out'
os.makedirs(WORK, exist_ok=True)
print('엑셀 있음:', os.path.exists(XLSX))

## 2. 참가자 엑셀 올리기
`TDBRAIN_participants_V3.1.xlsx`를 처음 한 번만 올리면 Drive에 저장되어 다음부터는 건너뜁니다.

In [ ]:
if not os.path.exists(XLSX):
    from google.colab import files
    import shutil
    up = files.upload()                       # TDBRAIN_participants_V3.1.xlsx 선택
    shutil.move(list(up)[0], XLSX)
print('엑셀 있음:', os.path.exists(XLSX))

## 3. 스크립트 파일 만들기

In [ ]:
%%writefile /content/tdbrain_cbramod_mdd.py
"""
TDBRAIN MDD vs HC — frozen CBraMod + L2 logistic regression baseline
(Mumtaz 실험과 동일한 설정을 TDBRAIN에 맞춰 옮긴 버전)

단계
  1) select   : participants xlsx에서 HC / MDD 피험자 선정 (성인, 첫 세션, 코호트별)
  2) extract  : EO/EC 안정 상태 EEG 전처리 -> frozen CBraMod 특징(3,800차원) 추출, 기록 단위로 캐시
  3) cv       : 피험자 단위 층화 5-fold, StandardScaler + LogisticRegression(C=0.01)

사용 예
  python tdbrain_cbramod_mdd.py select  --participants TDBRAIN_participants_V3.1.xlsx --out out
  python tdbrain_cbramod_mdd.py extract --participants TDBRAIN_participants_V3.1.xlsx --out out \
         --eeg-root D:/TDBRAIN --cbramod-repo D:/CBraMod --weights D:/CBraMod/pretrained_weights/pretrained_weights.pth
  python tdbrain_cbramod_mdd.py cv      --participants TDBRAIN_participants_V3.1.xlsx --out out --cohort matched
  (cohort: matched | full | formal)
  Colab에서 braindecode 가중치를 쓰려면 extract에서 --cbramod-repo / --weights 를 빼면 됨
"""
import argparse, glob, json, os, sys, time
import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------- 설정
SEED = 42
FS_TARGET = 200
SEG_SEC = 5
L_FREQ, H_FREQ, NOTCH = 0.3, 75.0, 50.0
AMP_DIV = 100.0              # μV / 100
FLAG_UV = 100.0              # 품질 검토용 표시 기준(제외에는 사용 안 함)
C_REG = 0.01
CONDITIONS = ["EO", "EC"]

# Mumtaz 19채널 순서 -> TDBRAIN(10-10) 채널명
MUMTAZ_ORDER = ["Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8", "T3", "C3", "Cz",
                "C4", "T4", "T5", "P3", "Pz", "P4", "T6", "O1", "O2"]
TO_TDBRAIN = {"T3": "T7", "T4": "T8", "T5": "P7", "T6": "P8"}
TDBRAIN_CH = [TO_TDBRAIN.get(c, c) for c in MUMTAZ_ORDER]


# ----------------------------------------------------------------------------- 1) 피험자 선정
def load_participants(path):
    d = pd.read_excel(path)
    d["ind"] = d["indication"].astype(str).str.strip().str.upper()
    d["fstat"] = d["formal_status"].astype(str).str.strip().str.upper()
    return d


def select_subjects(d, cohort="matched", min_age=18, caliper=3.0):
    """첫 세션 1개만 사용(다회차 세션 누수 방지), 성인만, 라벨 HC=0 / MDD=1."""
    d = d[d["ind"].isin(["HEALTHY", "MDD"])].copy()          # MDD 단독(동반 진단 표기 제외)
    d = d.sort_values("sessID").drop_duplicates("TDBRAIN_ID", keep="first")
    d = d[d["age"].notna() & (d["age"] >= min_age)]
    d["label"] = (d["ind"] == "MDD").astype(int)

    if cohort == "formal":                                    # MDD 중 formal_status도 MDD인 경우만
        d = d[(d.label == 0) | (d.fstat == "MDD")]
    elif cohort == "matched":                                 # HC 1명당 동일 성별·나이 최근접 MDD 1명
        rng = np.random.default_rng(SEED)
        hc = d[d.label == 0].sample(frac=1, random_state=SEED)
        pool = d[d.label == 1].copy()
        keep_hc, keep_mdd = [], []
        for _, r in hc.iterrows():
            cand = pool[pool.gender == r.gender]
            if cand.empty:
                continue
            diff = (cand.age - r.age).abs()
            if diff.min() > caliper:
                continue
            j = diff.idxmin()
            keep_hc.append(r.TDBRAIN_ID); keep_mdd.append(pool.loc[j, "TDBRAIN_ID"])
            pool = pool.drop(j)
        d = d[d.TDBRAIN_ID.isin(keep_hc + keep_mdd)]
    elif cohort != "full":
        raise ValueError(cohort)
    return d.reset_index(drop=True)


def describe(d):
    rows = []
    for lab, g in d.groupby("label"):
        rows.append({"group": "MDD" if lab else "HC", "n": len(g),
                     "age_mean": round(g.age.mean(), 1), "age_sd": round(g.age.std(), 1),
                     "gender=1": int((g.gender == 1).sum()), "gender=0": int((g.gender == 0).sum())})
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------- 2) 전처리 + 특징 추출
def find_file(eeg_root, sid, sess, cond):
    pat = os.path.join(eeg_root, "**", f"{sid}_ses-{int(sess)}_task-rest{cond}_eeg.*")
    hits = [h for h in glob.glob(pat, recursive=True) if h.lower().endswith((".bdf", ".vhdr", ".edf", ".csv"))]
    # 원본(V3.1은 .bdf) 우선, derivatives는 이미 전처리가 들어가 있으므로 차선
    hits.sort(key=lambda h: ("derivatives" in h.lower(), h.lower().endswith(".csv")))
    return hits[0] if hits else None


def load_raw(path):
    import mne
    low = path.lower()
    if low.endswith(".bdf"):
        raw = mne.io.read_raw_bdf(path, preload=True, verbose="ERROR")
    elif low.endswith(".vhdr"):
        raw = mne.io.read_raw_brainvision(path, preload=True, verbose="ERROR")
    elif low.endswith(".edf"):
        raw = mne.io.read_raw_edf(path, preload=True, verbose="ERROR")
    else:  # TDBRAIN derivatives csv (500 Hz, μV) -> MNE는 V 단위이므로 1e-6 곱해 통일
        df = pd.read_csv(path)
        names = [c for c in df.columns]
        info = mne.create_info(names, 500.0, ch_types="eeg")
        raw = mne.io.RawArray(df.values.T * 1e-6, info, verbose="ERROR")
    lut = {c.lower(): c for c in raw.ch_names}
    missing = [c for c in TDBRAIN_CH if c.lower() not in lut]
    if missing:
        raise RuntimeError(f"missing channels {missing} in {path}")
    raw.pick([lut[c.lower()] for c in TDBRAIN_CH])           # Mumtaz 순서로 정렬
    return raw


def preprocess(path):
    """reference 유지, 0.3–75 Hz, 50 Hz notch, 200 Hz, 5 s 비중첩, μV/100 -> [N,19,5,200]"""
    raw = load_raw(path)
    fs_orig = raw.info["sfreq"]
    raw.filter(L_FREQ, H_FREQ, verbose="ERROR")
    raw.notch_filter(NOTCH, verbose="ERROR")
    raw.resample(FS_TARGET, verbose="ERROR")
    x = raw.get_data() * 1e6                                  # V -> μV
    n = x.shape[1] // (FS_TARGET * SEG_SEC)                   # 잔여 구간 버림
    x = x[:, : n * FS_TARGET * SEG_SEC]
    segs = x.reshape(19, n, SEG_SEC, FS_TARGET).transpose(1, 0, 2, 3)   # [N,19,5,200]
    flag = (np.abs(segs).reshape(n, -1).max(1) > FLAG_UV)
    return (segs / AMP_DIV).astype(np.float32), flag, fs_orig, x.shape[1] / FS_TARGET


class _BraindecodeWrap:
    """braindecode CBraMod는 [B,19,1000] 입력 -> [B,19,5,200] 출력. 원본 repo와 같은 인터페이스로 맞춤."""
    def __init__(self, m): self.m = m
    def parameters(self): return self.m.parameters()
    def __call__(self, x): return self.m(x.reshape(x.shape[0], x.shape[1], -1))


def build_cbramod(repo, weights):
    import torch
    if repo is None:                                          # Colab: braindecode 사전학습 가중치 사용
        from braindecode.models import CBraMod
        m = CBraMod.from_pretrained("braindecode/cbramod-pretrained", return_encoder_output=True).eval()
        m = m.to("cuda" if torch.cuda.is_available() else "cpu")
        for p in m.parameters():
            p.requires_grad_(False)
        return _BraindecodeWrap(m)
    sys.path.insert(0, repo)
    from models.cbramod import CBraMod
    m = CBraMod(in_dim=200, out_dim=200, d_model=200, dim_feedforward=800,
                seq_len=30, n_layer=12, nhead=8)
    m.load_state_dict(torch.load(weights, map_location="cpu"))
    m.proj_out = torch.nn.Identity()
    m = m.to("cuda" if torch.cuda.is_available() else "cpu").eval()
    for p in m.parameters():
        p.requires_grad_(False)
    return m


def extract_features(model, segs, bs=64):
    import torch
    dev = next(model.parameters()).device
    out = []
    with torch.no_grad():
        for i in range(0, len(segs), bs):
            z = model(torch.from_numpy(segs[i:i + bs]).to(dev))   # [B,19,5,200]
            out.append(z.mean(dim=2).reshape(z.shape[0], -1).cpu().numpy())   # 시간 패치 평균 -> 3,800
    return np.concatenate(out).astype(np.float32)


def run_extract(a):
    d = select_subjects(load_participants(a.participants), "full")   # 모든 코호트의 합집합
    if a.limit:                                               # 시험 실행: HC/MDD 각각 앞에서 limit명
        d = pd.concat([g.head(a.limit) for _, g in d.groupby("label")])
    fdir = os.path.join(a.out, "features"); os.makedirs(fdir, exist_ok=True)
    model = build_cbramod(a.cbramod_repo, a.weights)
    print("device:", next(model.parameters()).device)
    log, t0 = [], time.time()
    for _, r in d.iterrows():
        for cond in CONDITIONS:
            fp = os.path.join(fdir, f"{r.TDBRAIN_ID}_{cond}.npz")
            rec = {"subject": r.TDBRAIN_ID, "label": int(r.label), "cond": cond}
            if os.path.exists(fp):
                z = np.load(fp); rec.update(status="cached", n_seg=len(z["feat"]), n_flag=int(z["flag"].sum()))
                log.append(rec); continue
            path = find_file(a.eeg_root, r.TDBRAIN_ID, r.sessID, cond)
            if path is None:
                rec.update(status="no_file"); log.append(rec); continue
            try:
                segs, flag, fs, dur = preprocess(path)
                if len(segs) == 0:
                    rec.update(status="too_short"); log.append(rec); continue
                feat = extract_features(model, segs)
                np.savez_compressed(fp, feat=feat, flag=flag)
                rec.update(status="ok", file=path, fs_orig=fs, dur_s=round(dur, 1),
                           n_seg=len(segs), n_flag=int(flag.sum()))
            except Exception as e:
                rec.update(status=f"error: {e}")
            log.append(rec)
            print(rec, flush=True)
    log = pd.DataFrame(log)
    log.to_csv(os.path.join(a.out, "extract_log.csv"), index=False)
    print(log.status.str.split(":").str[0].value_counts().to_string())
    print(f"done in {(time.time() - t0) / 60:.1f} min")


# ----------------------------------------------------------------------------- 3) 교차검증
def metrics(y, p, thr=0.5):
    from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score, f1_score, confusion_matrix
    yh = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0, 1]).ravel()
    return {"AUROC": roc_auc_score(y, p) if len(set(y)) > 1 else np.nan,
            "BalAcc": balanced_accuracy_score(y, yh), "Acc": accuracy_score(y, yh),
            "Sens": tp / (tp + fn) if tp + fn else np.nan, "Spec": tn / (tn + fp) if tn + fp else np.nan,
            "F1": f1_score(y, yh, zero_division=0), "TN": tn, "FP": fp, "FN": fn, "TP": tp}


def load_feature_table(fdir, subjects):
    X, rows = [], []
    for sid in subjects:
        for cond in CONDITIONS:
            fp = os.path.join(fdir, f"{sid}_{cond}.npz")
            if os.path.exists(fp):
                f = np.load(fp)["feat"]; X.append(f)
                rows += [(sid, cond)] * len(f)
    return np.concatenate(X), pd.DataFrame(rows, columns=["subject", "cond"])


def subject_scores(meta, prob):
    m = meta.assign(p=prob)
    per_cond = m.groupby(["subject", "cond"]).p.mean()       # 조건별 평균
    return per_cond.groupby("subject").mean()                 # EO/EC 동일 가중 (한 조건만 있으면 그 값)


def run_cv(a, X=None, meta=None, d=None):
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    if d is None:
        d = select_subjects(load_participants(a.participants), a.cohort)
    if X is None:
        X, meta = load_feature_table(os.path.join(a.out, "features"), d.TDBRAIN_ID)
    d = d[d.TDBRAIN_ID.isin(meta.subject.unique())].reset_index(drop=True)   # EEG 있는 피험자만
    lab = dict(zip(d.TDBRAIN_ID, d.label))
    keep = meta.subject.isin(lab).values
    X, meta = X[keep], meta[keep].reset_index(drop=True)
    y_seg = meta.subject.map(lab).values

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof_seg = np.full(len(meta), np.nan); fold_rows = []; subj_fold = {}
    for k, (tr, te) in enumerate(skf.split(d.TDBRAIN_ID, d.label), 1):
        tr_s, te_s = set(d.TDBRAIN_ID[tr]), set(d.TDBRAIN_ID[te])
        itr, ite = meta.subject.isin(tr_s).values, meta.subject.isin(te_s).values
        sc = StandardScaler().fit(X[itr])                     # 학습 fold로만 fit
        clf = LogisticRegression(C=C_REG, max_iter=5000).fit(sc.transform(X[itr]), y_seg[itr])
        oof_seg[ite] = clf.predict_proba(sc.transform(X[ite]))[:, 1]
        s = subject_scores(meta[ite], oof_seg[ite])
        fm = metrics(s.index.map(lab).values, s.values)
        fold_rows.append({"fold": k, "n_train": len(tr_s), "n_test": len(te_s), **fm})
        subj_fold.update({sid: k for sid in te_s})

    s = subject_scores(meta, oof_seg)
    y_sub = s.index.map(lab).values
    res = {"cohort": a.cohort, "n_subjects": len(s), "n_HC": int((y_sub == 0).sum()), "n_MDD": int((y_sub == 1).sum()),
           "n_recordings": int(meta.groupby(["subject", "cond"]).ngroups), "n_segments": len(meta),
           "subject_level": metrics(y_sub, s.values), "segment_level": metrics(y_seg, oof_seg)}
    from sklearn.metrics import average_precision_score
    res["subject_level"]["AUPRC"] = average_precision_score(y_sub, s.values)
    folds = pd.DataFrame(fold_rows)

    od = os.path.join(a.out, f"cv_{a.cohort}"); os.makedirs(od, exist_ok=True)
    folds.to_csv(os.path.join(od, "folds.csv"), index=False)
    pd.DataFrame({"subject": s.index, "label": y_sub, "p_mdd": s.values,
                  "fold": s.index.map(subj_fold)}).to_csv(os.path.join(od, "subject_oof.csv"), index=False)
    meta.groupby(["cond"]).size().rename("n_seg").to_csv(os.path.join(od, "segments_by_cond.csv"))
    describe(d).to_csv(os.path.join(od, "demographics.csv"), index=False)
    with open(os.path.join(od, "summary.json"), "w") as f:
        json.dump(res, f, indent=2, default=float)

    sl = res["subject_level"]
    print(f"[{a.cohort}] subjects={res['n_subjects']} (HC {res['n_HC']}, MDD {res['n_MDD']}), "
          f"recordings={res['n_recordings']}, segments={res['n_segments']}")
    print(f"  subject OOF: AUROC {sl['AUROC']:.4f} | BalAcc {sl['BalAcc']*100:.2f}% | Acc {sl['Acc']*100:.2f}% | "
          f"Sens {sl['Sens']*100:.2f}% | Spec {sl['Spec']*100:.2f}% | F1 {sl['F1']*100:.2f}% | AUPRC {sl['AUPRC']:.4f}")
    print(f"  confusion: TN={sl['TN']} FP={sl['FP']} FN={sl['FN']} TP={sl['TP']}")
    print(f"  fold AUROC {folds.AUROC.mean():.4f} ± {folds.AUROC.std():.4f} | "
          f"fold BalAcc {folds.BalAcc.mean()*100:.2f} ± {folds.BalAcc.std()*100:.2f}%p")
    print(f"  segment OOF: AUROC {res['segment_level']['AUROC']:.4f} | BalAcc {res['segment_level']['BalAcc']*100:.2f}%")
    return res


def run_select(a):
    d0 = load_participants(a.participants)
    os.makedirs(a.out, exist_ok=True)
    for c in ["full", "formal", "matched"]:
        d = select_subjects(d0, c)
        d[["TDBRAIN_ID", "sessID", "label", "age", "gender", "formal_status"]].to_csv(
            os.path.join(a.out, f"subjects_{c}.csv"), index=False)
        print(f"== {c}\n{describe(d).to_string(index=False)}\n")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("step", choices=["select", "extract", "cv"])
    ap.add_argument("--participants", required=True)
    ap.add_argument("--out", default="out_tdbrain")
    ap.add_argument("--eeg-root"); ap.add_argument("--cbramod-repo"); ap.add_argument("--weights")
    ap.add_argument("--limit", type=int, default=0, help="extract 시험 실행용 (그룹당 인원)")
    ap.add_argument("--cohort", default="matched", choices=["matched", "full", "formal"])
    a = ap.parse_args()
    {"select": run_select, "extract": run_extract, "cv": run_cv}[a.step](a)

## 4. 피험자 선정 확인
matched가 **HC 159 / MDD 159**로 나오면 정상

In [ ]:
!python /content/tdbrain_cbramod_mdd.py select --participants "$XLSX" --out "$OUT"

## 5. EEG 데이터 준비 (HC + MDD만 압축 해제)
Drive에 이미 풀어 둔 데이터가 있으면 로컬로 복사만 하고, 없으면 zip을 받아서 HC·MDD 피험자만 풉니다 (다운로드 약 15 GB, 끊기면 이어받기).
비밀번호는 TDBRAIN 다운로드 안내에 있는 zip 비밀번호를 입력하세요. **비밀번호를 코드에 적어 두지 마세요.**

In [ ]:
import pandas as pd, subprocess, getpass, os
DRIVE_DATA = f'{WORK}/TDBRAIN_Dataset_V3_1'
if os.path.isdir(DRIVE_DATA) and len(os.listdir(DRIVE_DATA)) > 400:
    print('Drive에 데이터 있음 -> 로컬로 복사'); os.makedirs(LOCAL, exist_ok=True)
    !cp -r "$DRIVE_DATA" "$LOCAL/"
else:
    URL = 'https://brainclinics.com/images/downloads/TDBRAIN_Dataset_V3_1_Encr.zip'
    for i in range(10):                       # 끊기면 이어받기
        !aria2c -x 8 -s 8 -c --max-tries=0 --retry-wait=5 --console-log-level=warn --summary-interval=60 -d /content -o tdbrain.zip "$URL"
        if subprocess.run(['unzip', '-l', '/content/tdbrain.zip'], capture_output=True).returncode == 0:
            break
        print(f'시도 {i + 1}: 아직 덜 받음, 이어받기 계속')
    df = pd.read_excel(XLSX)
    ind = df.indication.astype(str).str.strip().str.upper()
    ids = sorted(df[ind.isin(['HEALTHY', 'MDD'])].TDBRAIN_ID.unique())
    print('대상 피험자:', len(ids))
    with open('/content/hcmdd_patterns.txt', 'w') as f:
        f.write('\n'.join(f'TDBRAIN_Dataset_V3_1/{s}/*' for s in ids))
    PW = getpass.getpass('zip password: ')
    r = subprocess.run(['7z', 'x', f'-p{PW}', '/content/tdbrain.zip', f'-o{LOCAL}',
                        '-ir@/content/hcmdd_patterns.txt', '-y'], capture_output=True, text=True)
    del PW
    print(r.stdout[-400:], r.stderr[-400:])
    print('Drive에 백업 중...')
    !cp -r "$LOCAL/TDBRAIN_Dataset_V3_1" "$WORK/"
EEG_ROOT = f'{LOCAL}/TDBRAIN_Dataset_V3_1'
print('피험자 폴더 수:', len(os.listdir(EEG_ROOT)))   # 약 497이면 정상

확인 후 zip 삭제 (선택, 공간 확보용)

In [ ]:
!rm -f /content/tdbrain.zip

## 6. 원본 신호 끝부분 잡음 확인 (선택)
기록 끝부분에 큰 잡음이 몰려 있는지 10초 단위 최대 진폭으로 확인합니다.

In [ ]:
import mne, glob, numpy as np
mne.set_log_level('WARNING')
f = sorted(glob.glob(f'{EEG_ROOT}/sub-*/ses-1/eeg/*restEC_eeg.bdf'))[0]
raw = mne.io.read_raw_bdf(f, preload=True).pick(['Fp1','Fz','Cz','Pz','O1'])
raw.filter(0.3, 75).notch_filter(50)
x = raw.get_data() * 1e6
sec = np.arange(x.shape[1]) / raw.info['sfreq']
print(f, '| 길이(초):', round(sec[-1], 1))
for t0 in range(0, int(sec[-1]), 10):
    seg = x[:, (sec >= t0) & (sec < t0 + 10)]
    print(f'{t0:4d}-{t0+10:4d}s  max|μV| = {np.abs(seg).max():8.1f}')

## 7. CBraMod 원본 코드 + 사전학습 가중치
`wjq-learning/CBraMod` 원본 코드와 `weighting666/CBraMod`의 `pretrained_weights.pth`를 씁니다.

In [ ]:
!git clone -q https://github.com/wjq-learning/CBraMod.git /content/CBraMod
!pip install -q einops
import os
W = '/content/CBraMod/pretrained_weights/pretrained_weights.pth'
if not os.path.exists(W):                      # repo에 없으면 Hugging Face에서 받기
    from huggingface_hub import hf_hub_download
    W = hf_hub_download('weighting666/CBraMod', 'pretrained_weights.pth')
print(W, round(os.path.getsize(W) / 1e6, 1), 'MB')

In [ ]:
import sys, torch
sys.path.insert(0, '/content')
import importlib, tdbrain_cbramod_mdd as t; importlib.reload(t)
m = t.build_cbramod('/content/CBraMod', W)
x = torch.randn(2, 19, 5, 200).to(next(m.parameters()).device)
with torch.no_grad():
    print(m(x).shape)      # torch.Size([2, 19, 5, 200]) 이면 정상

## 8. 특징 추출 — 시험 실행 (그룹당 3명)
에러 없이 `ok`가 나오고 `n_seg`가 조건당 약 24개면 다음 셀로

In [ ]:
!python /content/tdbrain_cbramod_mdd.py extract --participants "$XLSX" --out "$OUT" --eeg-root "$EEG_ROOT" --cbramod-repo /content/CBraMod --weights "$W" --limit 3

## 9. 특징 추출 — 전체
특징은 Drive에 기록 단위로 저장되므로, 런타임이 끊겨도 다시 실행하면 이어서 진행됩니다.

In [ ]:
!python /content/tdbrain_cbramod_mdd.py extract --participants "$XLSX" --out "$OUT" --eeg-root "$EEG_ROOT" --cbramod-repo /content/CBraMod --weights "$W"

In [ ]:
import pandas as pd
log = pd.read_csv(f'{OUT}/extract_log.csv')
print(log.status.str.split(':').str[0].value_counts(), '\n')
print(log.groupby(['label', 'cond'])[['n_seg', 'n_flag']].agg(['count', 'mean', 'sum']))
log[~log.status.isin(['ok', 'cached'])]

## 10. 실험 1: EEG-only 분류
주 결과 = matched (나이·성별 1:1), 보조 = full

In [ ]:
!python /content/tdbrain_cbramod_mdd.py cv --participants "$XLSX" --out "$OUT" --cohort matched

In [ ]:
!python /content/tdbrain_cbramod_mdd.py cv --participants "$XLSX" --out "$OUT" --cohort full

## 11. 실험 2: 조건별 단독 평가와 잡음 점검
EO만 / EC만으로 같은 평가를 반복하고, 그룹별 100 μV 초과 구간 비율을 비교합니다.
그룹 간 잡음 비율 차이가 작으면 성능이 잡음 양 차이에서 온 것이 아니라는 근거가 됩니다.

In [ ]:
import sys, types, importlib
sys.path.insert(0, '/content')
import tdbrain_cbramod_mdd as t; importlib.reload(t)

d = t.select_subjects(t.load_participants(XLSX), 'matched')
X, meta = t.load_feature_table(f'{OUT}/features', d.TDBRAIN_ID)
for cond in ['EO', 'EC']:
    k = (meta.cond == cond).values
    t.run_cv(types.SimpleNamespace(cohort=f'matched_{cond}only', out=OUT),
             X[k], meta[k].reset_index(drop=True), d)

In [ ]:
log = pd.read_csv(f'{OUT}/extract_log.csv')
log['flag_rate'] = log.n_flag / log.n_seg
log.groupby(['cond', 'label']).flag_rate.describe().round(3)

## 12. 실험 3: 임상정보 조합
EEG 점수(EEG-only 모델의 MDD 확률)와 임상 변수를 결합합니다. 학습 fold의 EEG 점수는 내부 5-fold로 다시 만들어 누수를 막습니다(처음 한 번 계산 후 캐시).
- `clinical_combo.py`: 맥락 8개 + well의 모든 조합(511개), 중첩 선택, EEG 비중, 누출 대조(BDI·rTMS·NEO-FFI 기록 유무)
- `swap_experiment.py`: 실험 4 (실험 3의 EEG 점수 캐시를 사용하므로 실험 3을 먼저 실행)

In [ ]:
%%writefile /content/clinical_combo.py
"""
TDBRAIN MDD vs HC — EEG-only vs EEG + 임상정보 조합 비교
(tdbrain_cbramod_mdd.py 로 추출한 CBraMod 특징을 그대로 사용)

1) EEG 점수: 기존 baseline과 동일 (구간 단위 StandardScaler + LR(C=0.01), EO/EC 동일 가중 평균)
   - 평가 fold 점수  = 학습 fold 전체로 학습한 EEG 모델의 예측 (= EEG-only baseline과 동일)
   - 학습 fold 점수  = 학습 fold 안에서 다시 5-fold로 만든 out-of-fold 예측 (스태킹 누수 방지)
2) 임상 변수 조합마다
   - Clinical-only : 임상 변수만으로 LR
   - Fusion        : [EEG 점수(logit)] + 임상 변수 -> LR (late fusion / stacking)
3) 중첩 선택: 학습 fold 안에서만 최적 조합을 고르고 평가 fold에서 성능을 잼 (선택 편향 없는 '최상 조합' 성능)
4) 누출 대조 실험: BDI_pre / rTMS / NEO-FFI '값 있음' 여부를 넣었을 때

사용:
  python clinical_combo.py --participants XLSX --out OUT --cohort matched
"""
import argparse, itertools, json, os, sys, time
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import tdbrain_cbramod_mdd as base

SEED = 42
CONTEXT = ["age", "gender", "education", "sleep", "coffee", "cigarette", "beer", "meal"]   # 맥락 그룹
SYMPTOM = ["well"]                                                                          # 증상 근접
LEAK = ["has_BDI", "has_rTMS", "has_NEO"]                                                   # 누출 대조
C_META = 1.0


# ----------------------------------------------------------------------------- 임상 변수 정리
def clinical_table(d):
    c = pd.DataFrame(index=d.TDBRAIN_ID.values)
    for v in CONTEXT + SYMPTOM:
        x = pd.to_numeric(d[v], errors="coerce").values.astype(float)
        x[x == 99] = np.nan                          # 99 = 결측 코드
        c[v] = x
    c.loc[c.sleep > 24, "sleep"] = np.nan            # sleep=205 같은 오류값
    c["has_BDI"] = d["BDI_pre"].notna().astype(float).values
    c["has_rTMS"] = d["rTMS PROTOCOL"].notna().astype(float).values
    c["has_NEO"] = d["neoFFI_q1"].notna().astype(float).values
    return c


# ----------------------------------------------------------------------------- EEG 점수 (스태킹용)
def seg_model_scores(X, meta, lab, tr_subj, te_subj):
    itr = meta.subject.isin(tr_subj).values
    ite = meta.subject.isin(te_subj).values
    y = meta.subject[itr].map(lab).values
    sc = StandardScaler().fit(X[itr])
    clf = LogisticRegression(C=base.C_REG, max_iter=5000).fit(sc.transform(X[itr]), y)
    p = clf.predict_proba(sc.transform(X[ite]))[:, 1]
    return base.subject_scores(meta[ite].reset_index(drop=True), p)      # Series: subject -> prob


def eeg_scores(X, meta, d, cache):
    if os.path.exists(cache):
        return json.load(open(cache))
    lab = dict(zip(d.TDBRAIN_ID, d.label))
    subj, y = d.TDBRAIN_ID.values, d.label.values
    outer = StratifiedKFold(5, shuffle=True, random_state=SEED)
    folds = []
    for k, (tr, te) in enumerate(outer.split(subj, y), 1):
        t0 = time.time()
        s_te = seg_model_scores(X, meta, lab, set(subj[tr]), set(subj[te]))
        inner = StratifiedKFold(5, shuffle=True, random_state=SEED)
        s_tr, inner_splits = {}, []
        for itr, iva in inner.split(subj[tr], y[tr]):
            s = seg_model_scores(X, meta, lab, set(subj[tr][itr]), set(subj[tr][iva]))
            s_tr.update(s.to_dict())
            inner_splits.append([subj[tr][itr].tolist(), subj[tr][iva].tolist()])
        folds.append({"train": subj[tr].tolist(), "test": subj[te].tolist(),
                      "eeg_train": s_tr, "eeg_test": s_te.to_dict(), "inner": inner_splits})
        print(f"  EEG fold {k}/5 done ({time.time() - t0:.0f}s)", flush=True)
    json.dump(folds, open(cache, "w"))
    return folds


# ----------------------------------------------------------------------------- 조합 평가
def logit(p):
    p = np.clip(np.asarray(p, float), 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))


def design(ids, clin, cols, eeg=None):
    parts = []
    if eeg is not None:
        parts.append(logit([eeg[i] for i in ids])[:, None])
    if cols:
        parts.append(clin.loc[ids, cols].values)
    return np.hstack(parts)


def meta_model():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(C=C_META, max_iter=5000))


def fit_predict(tr_ids, te_ids, clin, cols, lab, eeg_tr=None, eeg_te=None):
    Xtr = design(tr_ids, clin, cols, eeg_tr)
    Xte = design(te_ids, clin, cols, eeg_te)
    m = meta_model().fit(Xtr, [lab[i] for i in tr_ids])
    w = m[-1].coef_[0]
    return m.predict_proba(Xte)[:, 1], w


def evaluate(folds, clin, cols, lab, use_eeg):
    ids, ps, w_eeg_share = [], [], []
    for f in folds:
        p, w = fit_predict(f["train"], f["test"], clin, cols, lab,
                           f["eeg_train"] if use_eeg else None, f["eeg_test"] if use_eeg else None)
        ids += f["test"]; ps += list(p)
        if use_eeg and cols:
            w_eeg_share.append(abs(w[0]) / np.abs(w).sum())
    y = np.array([lab[i] for i in ids]); ps = np.array(ps)
    return {"AUROC": roc_auc_score(y, ps), "BalAcc": balanced_accuracy_score(y, ps >= 0.5),
            "EEG_weight_share": float(np.mean(w_eeg_share)) if w_eeg_share else (1.0 if use_eeg else 0.0)}


def inner_auc(f, clin, cols, lab):
    """학습 fold 안에서만: inner split으로 fusion 성능 측정 (조합 선택용)"""
    ids, ps = [], []
    for itr, iva in f["inner"]:
        p, _ = fit_predict(itr, iva, clin, cols, lab, f["eeg_train"], f["eeg_train"])
        ids += iva; ps += list(p)
    return roc_auc_score([lab[i] for i in ids], ps)


def nested_best(folds, clin, subsets, lab):
    ids, ps, chosen = [], [], []
    for f in folds:
        scores = [(inner_auc(f, clin, list(s), lab), s) for s in subsets]
        best = max(scores, key=lambda t: t[0])[1]
        p, _ = fit_predict(f["train"], f["test"], clin, list(best), lab, f["eeg_train"], f["eeg_test"])
        ids += f["test"]; ps += list(p); chosen.append("+".join(best) or "(EEG only)")
    y = np.array([lab[i] for i in ids]); ps = np.array(ps)
    return {"AUROC": roc_auc_score(y, ps), "BalAcc": balanced_accuracy_score(y, ps >= 0.5), "chosen_per_fold": chosen}


# ----------------------------------------------------------------------------- main
def main(a):
    d = base.select_subjects(base.load_participants(a.participants), a.cohort)
    X, meta = base.load_feature_table(os.path.join(a.out, "features"), d.TDBRAIN_ID)
    d = d[d.TDBRAIN_ID.isin(meta.subject.unique())].reset_index(drop=True)
    keep = meta.subject.isin(d.TDBRAIN_ID).values
    X, meta = X[keep], meta[keep].reset_index(drop=True)
    lab = dict(zip(d.TDBRAIN_ID, d.label))
    clin = clinical_table(d)
    od = os.path.join(a.out, f"clinical_{a.cohort}"); os.makedirs(od, exist_ok=True)

    print(f"[{a.cohort}] subjects={len(d)} (HC {(d.label == 0).sum()}, MDD {(d.label == 1).sum()})")
    print("EEG 점수 계산 (처음 한 번만, 이후 캐시 사용)...")
    folds = eeg_scores(X, meta, d, os.path.join(od, "eeg_scores.json"))

    eeg_only = evaluate(folds, clin, [], lab, use_eeg=True)
    print(f"EEG-only (baseline 재현 확인): AUROC {eeg_only['AUROC']:.4f} | BalAcc {eeg_only['BalAcc'] * 100:.2f}%")

    pool = CONTEXT + SYMPTOM
    subsets = [s for r in range(1, len(pool) + 1) for s in itertools.combinations(pool, r)]
    rows = []
    t0 = time.time()
    for s in subsets:
        cols = list(s)
        co = evaluate(folds, clin, cols, lab, use_eeg=False)
        fu = evaluate(folds, clin, cols, lab, use_eeg=True)
        rows.append({"variables": "+".join(cols), "n_vars": len(cols), "has_well": "well" in cols,
                     "clin_AUROC": co["AUROC"], "clin_BalAcc": co["BalAcc"],
                     "fusion_AUROC": fu["AUROC"], "fusion_BalAcc": fu["BalAcc"],
                     "dAUROC_vs_EEG": fu["AUROC"] - eeg_only["AUROC"],
                     "dBalAcc_vs_EEG": fu["BalAcc"] - eeg_only["BalAcc"],
                     "EEG_weight_share": fu["EEG_weight_share"]})
    tab = pd.DataFrame(rows).sort_values("fusion_AUROC", ascending=False)
    tab.to_csv(os.path.join(od, "all_combinations.csv"), index=False)
    print(f"{len(subsets)}개 조합 평가 완료 ({time.time() - t0:.0f}s)")

    # 단일 변수 효과
    single = tab[tab.n_vars == 1].sort_values("dAUROC_vs_EEG", ascending=False)
    single.to_csv(os.path.join(od, "single_variables.csv"), index=False)

    # 중첩 선택 (맥락만 / 맥락+well)
    ctx_subsets = [()] + [s for s in subsets if "well" not in s]
    all_subsets = [()] + subsets
    print("중첩 선택 중 (조합 선택을 학습 fold 안에서만)...")
    nb_ctx = nested_best(folds, clin, ctx_subsets, lab)
    nb_all = nested_best(folds, clin, all_subsets, lab)

    # 누출 대조
    leak_rows = []
    for name, cols in [("EEG + has_BDI", ["has_BDI"]), ("EEG + has_rTMS", ["has_rTMS"]),
                       ("EEG + has_NEO", ["has_NEO"]), ("EEG + 누출 3개", LEAK)]:
        fu = evaluate(folds, clin, cols, lab, use_eeg=True)
        co = evaluate(folds, clin, cols, lab, use_eeg=False)
        leak_rows.append({"setting": name, "clin_only_AUROC": co["AUROC"], "fusion_AUROC": fu["AUROC"],
                          "fusion_BalAcc": fu["BalAcc"], "EEG_weight_share": fu["EEG_weight_share"]})
    leak = pd.DataFrame(leak_rows)
    leak.to_csv(os.path.join(od, "leakage_control.csv"), index=False)

    summary = {"cohort": a.cohort, "n_subjects": len(d), "EEG_only": eeg_only,
               "best_fixed_context": tab[~tab.has_well].iloc[0].to_dict(),
               "best_fixed_any": tab.iloc[0].to_dict(),
               "nested_best_context": nb_ctx, "nested_best_with_well": nb_all}
    json.dump(summary, open(os.path.join(od, "summary.json"), "w"), indent=2, default=float)

    pd.set_option("display.width", 200)
    f = lambda t: t.round(4).to_string(index=False)
    print("\n=== 상위 10개 조합 (맥락 변수만) ===")
    print(f(tab[~tab.has_well].head(10)[["variables", "clin_AUROC", "fusion_AUROC", "fusion_BalAcc", "dAUROC_vs_EEG", "EEG_weight_share"]]))
    print("\n=== 상위 10개 조합 (well 포함 가능) ===")
    print(f(tab.head(10)[["variables", "clin_AUROC", "fusion_AUROC", "fusion_BalAcc", "dAUROC_vs_EEG", "EEG_weight_share"]]))
    print("\n=== 단일 변수 효과 ===")
    print(f(single[["variables", "clin_AUROC", "fusion_AUROC", "dAUROC_vs_EEG", "EEG_weight_share"]]))
    print("\n=== 중첩 선택 (선택 편향 없는 '최상 조합' 성능) ===")
    for nm, r in [("맥락만", nb_ctx), ("맥락+well", nb_all)]:
        print(f"{nm:8s} AUROC {r['AUROC']:.4f} | BalAcc {r['BalAcc'] * 100:.2f}% | fold별 선택: {r['chosen_per_fold']}")
    print("\n=== 누출 대조 실험 ===")
    print(f(leak))
    print(f"\n결과 저장: {od}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--participants", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--cohort", default="matched", choices=["matched", "full", "formal"])
    main(ap.parse_args())

In [ ]:
%%writefile /content/swap_experiment.py
"""
정보 교체 실험 (계획서 2단계: 동일한 EEG에 서로 다른 임상정보 제공)

학습된 fusion 모델([EEG 점수] + 임상 변수 -> LR)에, 평가 fold 피험자의 EEG 점수는 그대로 두고
임상정보만 바꿔 넣어 예측이 얼마나 흔들리는지 측정한다.

조건
  1 none      : 임상정보 없음 (EEG-only 모델)
  2 real      : 본인 임상정보
  3 random    : 무작위 다른 피험자의 임상정보
  4 same      : 같은 그룹(HC/MDD)의 다른 피험자 임상정보
  5 opposite  : 반대 그룹 피험자의 임상정보

모델
  EEG + 맥락 8개 / EEG + 맥락 8개 + well / EEG + 누출 3개 / EEG + 맥락 8개 + 누출 3개

지표 (조건 3~5는 기증자를 무작위로 R번 뽑아 평균)
  AUROC, BalAcc
  flip_vs_real   : 본인 정보일 때와 예측(MDD/HC)이 달라진 비율
  dP_vs_real     : 본인 정보 대비 MDD 확률 변화량 평균 |Δp|
  flip_vs_EEG    : EEG-only 예측과 달라진 비율
  (opposite) HC->MDD : HC 뇌파 + MDD 차트일 때 MDD로 판정된 비율
             MDD->HC : MDD 뇌파 + HC 차트일 때 HC로 판정된 비율
             follow_text : EEG-only 판정과 차트 그룹이 충돌할 때 차트 쪽을 따른 비율

사용:
  python swap_experiment.py --participants XLSX --out OUT --cohort matched
  (먼저 clinical_combo.py 를 같은 cohort로 한 번 실행해서 EEG 점수 캐시가 있어야 함)
"""
import argparse, json, os, sys
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import tdbrain_cbramod_mdd as base
import clinical_combo as cc

R = 20          # 기증자 무작위 추출 반복 횟수
SEED = 42
MODELS = {
    "EEG + 맥락8": cc.CONTEXT,
    "EEG + 맥락8 + well": cc.CONTEXT + cc.SYMPTOM,
    "EEG + 누출3": cc.LEAK,
    "EEG + 맥락8 + 누출3": cc.CONTEXT + cc.LEAK,
}


def fit_meta(f, clin, cols, lab):
    tr = f["train"]
    m = cc.meta_model().fit(cc.design(tr, clin, cols, f["eeg_train"]), [lab[i] for i in tr])
    return m


def predict(m, ids, clin_rows, cols, eeg):
    """clin_rows: ids 순서대로 넣을 임상정보 (DataFrame, index=ids)"""
    return m.predict_proba(cc.design(ids, clin_rows, cols, eeg))[:, 1]


def donors(ids, lab, all_ids, mode, rng):
    out = []
    by = {0: [i for i in all_ids if lab[i] == 0], 1: [i for i in all_ids if lab[i] == 1]}
    for i in ids:
        if mode == "random":
            pool = [j for j in all_ids if j != i]
        elif mode == "same":
            pool = [j for j in by[lab[i]] if j != i]
        else:  # opposite
            pool = by[1 - lab[i]]
        out.append(pool[rng.integers(len(pool))])
    return out


def main(a):
    d = base.select_subjects(base.load_participants(a.participants), a.cohort)
    od = os.path.join(a.out, f"clinical_{a.cohort}")
    cache = os.path.join(od, "eeg_scores.json")
    if not os.path.exists(cache):
        sys.exit(f"EEG 점수 캐시가 없습니다: {cache}\n먼저 clinical_combo.py --cohort {a.cohort} 를 실행하세요.")
    folds = json.load(open(cache))
    ids_all = [i for f in folds for i in f["test"]]
    d = d[d.TDBRAIN_ID.isin(ids_all)].reset_index(drop=True)
    lab = dict(zip(d.TDBRAIN_ID, d.label))
    clin = cc.clinical_table(d)
    rng = np.random.default_rng(SEED)
    print(f"[{a.cohort}] subjects={len(d)} (HC {(d.label == 0).sum()}, MDD {(d.label == 1).sum()}), donor 반복 R={R}")

    # EEG-only 기준 예측
    p_eeg = {}
    for f in folds:
        m = fit_meta(f, clin, [], lab)
        p_eeg.update(zip(f["test"], predict(m, f["test"], clin, [], f["eeg_test"])))

    rows, per_subject = [], []
    y_all = np.array([lab[i] for i in ids_all])
    pe = np.array([p_eeg[i] for i in ids_all])
    rows.append({"model": "EEG-only", "condition": "1 none", "AUROC": roc_auc_score(y_all, pe),
                 "BalAcc": balanced_accuracy_score(y_all, pe >= .5),
                 "flip_vs_real": 0.0, "dP_vs_real": 0.0, "flip_vs_EEG": 0.0})

    for name, cols in MODELS.items():
        models = [fit_meta(f, clin, cols, lab) for f in folds]
        # 2 real
        p_real = {}
        for f, m in zip(folds, models):
            p_real.update(zip(f["test"], predict(m, f["test"], clin, cols, f["eeg_test"])))
        pr = np.array([p_real[i] for i in ids_all])
        rows.append({"model": name, "condition": "2 real", "AUROC": roc_auc_score(y_all, pr),
                     "BalAcc": balanced_accuracy_score(y_all, pr >= .5), "flip_vs_real": 0.0, "dP_vs_real": 0.0,
                     "flip_vs_EEG": np.mean((pr >= .5) != (pe >= .5))})

        for cond in ["random", "same", "opposite"]:
            acc = {k: [] for k in ["AUROC", "BalAcc", "flip_vs_real", "dP_vs_real", "flip_vs_EEG",
                                   "HC->MDD", "MDD->HC", "follow_text"]}
            p_sum = np.zeros(len(ids_all))
            for r in range(R):
                p_sw = {}
                for f, m in zip(folds, models):
                    te = f["test"]
                    don = donors(te, lab, ids_all, cond, rng)
                    rows_sw = clin.loc[don].copy(); rows_sw.index = te          # 임상정보만 교체
                    p_sw.update(zip(te, predict(m, te, rows_sw, cols, f["eeg_test"])))
                ps = np.array([p_sw[i] for i in ids_all]); p_sum += ps
                acc["AUROC"].append(roc_auc_score(y_all, ps))
                acc["BalAcc"].append(balanced_accuracy_score(y_all, ps >= .5))
                acc["flip_vs_real"].append(np.mean((ps >= .5) != (pr >= .5)))
                acc["dP_vs_real"].append(np.mean(np.abs(ps - pr)))
                acc["flip_vs_EEG"].append(np.mean((ps >= .5) != (pe >= .5)))
                if cond == "opposite":
                    hc, mdd = y_all == 0, y_all == 1
                    acc["HC->MDD"].append(np.mean(ps[hc] >= .5))
                    acc["MDD->HC"].append(np.mean(ps[mdd] < .5))
                    chart = 1 - y_all                                   # 차트(기증자) 그룹
                    conflict = (pe >= .5).astype(int) != chart          # EEG 판정과 차트가 반대
                    acc["follow_text"].append(np.mean((ps[conflict] >= .5).astype(int) == chart[conflict]))
            row = {"model": name, "condition": {"random": "3 random", "same": "4 same", "opposite": "5 opposite"}[cond]}
            for k, v in acc.items():
                if v:
                    row[k] = float(np.mean(v))
            rows.append(row)
            if cond == "opposite":
                for i, p0, p1, p2 in zip(ids_all, pe, pr, p_sum / R):
                    per_subject.append({"model": name, "subject": i, "label": lab[i],
                                        "p_EEG_only": p0, "p_real": p1, "p_opposite_mean": p2})

    tab = pd.DataFrame(rows)
    tab.to_csv(os.path.join(od, "swap_results.csv"), index=False)
    pd.DataFrame(per_subject).to_csv(os.path.join(od, "swap_per_subject.csv"), index=False)

    pd.set_option("display.width", 220)
    show = ["model", "condition", "AUROC", "BalAcc", "flip_vs_real", "dP_vs_real", "flip_vs_EEG"]
    print("\n=== 조건별 성능과 예측 변화 ===")
    print(tab[show].round(4).to_string(index=False))
    print("\n=== 조건 5 (반대 그룹 임상정보): EEG와 차트가 충돌할 때 ===")
    opp = tab[tab.condition == "5 opposite"][["model", "HC->MDD", "MDD->HC", "follow_text"]]
    print(opp.round(4).to_string(index=False))
    print("\n  HC->MDD    : HC 뇌파 + MDD 환자 차트 -> MDD로 판정된 비율")
    print("  MDD->HC    : MDD 뇌파 + HC 차트 -> HC로 판정된 비율")
    print("  follow_text: EEG-only 판정과 차트 그룹이 반대일 때, 최종 판정이 차트를 따른 비율")
    print(f"\n결과 저장: {od}/swap_results.csv, swap_per_subject.csv")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--participants", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--cohort", default="matched", choices=["matched", "full", "formal"])
    main(ap.parse_args())

In [ ]:
!python /content/clinical_combo.py --participants "$XLSX" --out "$OUT" --cohort matched

In [ ]:
!python /content/clinical_combo.py --participants "$XLSX" --out "$OUT" --cohort full

## 13. 실험 4: 정보 교체
EEG 점수는 그대로 두고 임상정보만 본인 / 무작위 / 같은 그룹 / 반대 그룹 것으로 바꿔 넣습니다. 다른 환자는 20번 무작위로 뽑아 평균.

In [ ]:
!python /content/swap_experiment.py --participants "$XLSX" --out "$OUT" --cohort matched

In [ ]:
!python /content/swap_experiment.py --participants "$XLSX" --out "$OUT" --cohort full

## 14. 통합 스크립트로 재실행 (PD-EEG·AD-EEG와 같은 형식)
`tdbrain_fusion.py`는 PD-EEG·AD-EEG 스크립트와 같은 단계(`inspect / cv / clinical / swap / relate / preserve`)로 TDBRAIN을 다시 분석합니다.
이미 뽑아 둔 특징(`out/features`)을 그대로 쓰고, 결과는 `out_fusion`에 따로 저장합니다.

`tdbrain_fusion.py`는 AD-EEG에서 쓴 `ad_cbramod.py`(실험 1·3·4)와 `ad_extra.py`(추가 B·A)를 불러와 TDBRAIN 형식에 맞춰 실행하므로, 세 파일을 모두 만듭니다.

In [ ]:
%%writefile /content/ad_cbramod.py
"""
AD-EEG (OpenNeuro ds004504) — frozen CBraMod + L2 logistic regression
(TDBRAIN·Mumtaz·PD-EEG와 같은 프로토콜을 별도 과제로 적용)

과제 (--task)
  ad_cn  : AD(1) vs CN(0)      — 주 과제
  ad_ftd : AD(1) vs FTD(0)     — 두 치매 구분 (MMSE가 정답을 그대로 알려주지 않는 과제)
  ftd_cn : FTD(1) vs CN(0)
  all    : 특징 추출용 (88명 전원)

단계
  inspect  : participants.tsv 열·그룹별 값, 첫 EEG 파일의 채널·샘플링·진폭 확인, 임상 변수 분류 제안
  select   : 피험자 선정 (full, matched)
  extract  : 안정 상태(눈 감음) EEG 전처리 -> frozen CBraMod 특징(3,800차원), 기록 단위 캐시
  cv / seeds / noise / clinical / swap : 실험 1~4 (PD-EEG와 동일)

TDBRAIN·Mumtaz·PD-EEG와 다른 점
  - 안정 상태 기록이 눈 감은 상태(EC) 하나뿐
  - 19채널 10-20 (Mumtaz와 같은 이름·순서), 원본 500 Hz -> 200 Hz
  - 기록이 길다 (평균 약 13분 -> 기록당 약 150구간)
"""
import argparse, glob, itertools, json, os, re, sys, time
import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------- 설정 (TDBRAIN·Mumtaz와 동일)
SEED = 42
FS_TARGET = 200
SEG_SEC = 5
L_FREQ, H_FREQ, NOTCH = 0.3, 75.0, 50.0
AMP_DIV = 100.0              # μV / 100
FLAG_UV = 100.0              # 품질 검토용 표시 기준(제외에는 사용 안 함)
C_REG = 0.01                 # EEG 분류기
C_FUSION = 1.0               # 임상정보 결합 분류기 (TDBRAIN 실험 3과 동일)
N_SWAP = 20                  # 실험 4 기증자 추출 횟수
MAX_COMBO_VARS = 10          # 실험 3 조합에 쓸 최대 변수 수 (2^10-1 = 1,023개 조합)
CONDITIONS = ["EC"]          # 안정 상태 = 눈 감음
COG_CUT = 26                 # task=cog: PD 환자 중 MoCA < 26 을 인지 저하(1)로
N0, N1 = "CN", "AD"          # 라벨 0/1 이름 (과제에 따라 바뀜)
TASKS = {"ad_cn": ("C", "A", "CN", "AD"), "ftd_cn": ("C", "F", "CN", "FTD"), "ad_ftd": ("F", "A", "FTD", "AD")}

# Mumtaz 19채널 순서 (10-10 이름: T3->T7, T4->T8, T5->P7, T6->P8)
TARGET_19 = ["Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8", "T7", "C3", "Cz",
             "C4", "T8", "P7", "P3", "Pz", "P4", "P8", "O1", "O2"]
ALIAS = {"t3": "t7", "t4": "t8", "t5": "p7", "t6": "p8"}
SYMPTOM_KEYS = ["updrs", "hy", "hoehn", "yahr", "moca", "mmse", "bdi", "gds", "hads", "hamd", "depress",
                "anx", "ctt", "fog", "tremor", "ledd", "levodopa", "duration", "medic", "pdq", "nmss", "tmt", "tug"]


# ----------------------------------------------------------------------------- 참가자 표
def _find_col(d, pattern):
    for c in d.columns:
        if re.search(pattern, c, re.IGNORECASE):
            return c
    return None


def detect_label_col(d):
    for c in d.columns:
        if c == "participant_id":
            continue
        v = d[c].dropna().astype(str).str.strip().str.upper()
        if 1 < v.nunique() <= 6 and v.str.contains(r"PD|PARK").any() and v.str.contains(r"HC|CONTROL|HEALTHY|CTRL").any():
            return c
    return None


def load_participants(path, label_col=None, task="ad_cn"):
    """Group 열: A = AD, F = FTD, C = CN. 과제에 맞는 두 그룹만 남기고 label 0/1 부여"""
    global N0, N1
    d = pd.read_csv(path, sep="\t", na_values=["n/a", "N/A", "NA", "nan", "NaN", ""])
    d["subject"] = d["participant_id"].astype(str)
    gcol = label_col or _find_col(d, r"^group$")
    if gcol is None:
        raise RuntimeError(f"Group 열을 찾지 못했습니다. 열 목록: {list(d.columns)} -> --label-col 로 지정")
    g = d[gcol].astype(str).str.strip().str.upper()
    if task == "all":
        d = d.copy(); d["label"] = (g != "C").astype(int); N0, N1 = "CN", "치매"
    else:
        neg, pos, N0, N1 = TASKS[task]
        keep = g.isin([neg, pos])
        d = d[keep].copy(); d["label"] = (g[keep] == pos).astype(int)
    d.attrs["task"] = task
    d.attrs["label_col"] = gcol
    d.attrs["age_col"] = _find_col(d, r"^age")
    d.attrs["sex_col"] = _find_col(d, r"^(sex|gender)")
    return d.reset_index(drop=True)


def select_subjects(d, cohort="full", caliper=3.0):
    if cohort == "full":
        return d.copy()
    if cohort != "matched":
        raise ValueError(cohort)
    age, sex = d.attrs.get("age_col"), d.attrs.get("sex_col")
    if age is None or sex is None:
        raise RuntimeError("나이·성별 열이 없어 matched 코호트를 만들 수 없습니다. --cohort full 사용")
    dd = d[d[age].notna() & d[sex].notna()]
    hc = dd[dd.label == 0].sample(frac=1, random_state=SEED)
    pool = dd[dd.label == 1].copy()
    keep = []
    for _, r in hc.iterrows():                                # HC 1명당 같은 성별·나이 최근접 PD 1명
        cand = pool[pool[sex] == r[sex]]
        if cand.empty:
            continue
        diff = (cand[age] - r[age]).abs()
        if diff.min() > caliper:
            continue
        j = diff.idxmin()
        keep += [r.subject, pool.loc[j, "subject"]]
        pool = pool.drop(j)
    out = d[d.subject.isin(keep)].reset_index(drop=True)
    out.attrs = d.attrs
    return out


def describe(d):
    age, sex = d.attrs.get("age_col"), d.attrs.get("sex_col")
    rows = []
    for lab, g in d.groupby("label"):
        r = {"group": N1 if lab else N0, "n": len(g)}
        if age:
            r.update(age_mean=round(g[age].mean(), 1), age_sd=round(g[age].std(), 1))
        if sex:
            r.update({f"{sex}={k}": int(v) for k, v in g[sex].value_counts().items()})
        rows.append(r)
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------- EEG 읽기·전처리
def find_file(eeg_root, sid):
    hits = []
    for ext in ("set", "edf", "bdf", "vhdr", "fif"):
        hits += glob.glob(os.path.join(eeg_root, "**", f"{sid}_*task-*_eeg.{ext}"), recursive=True)
    hits.sort(key=lambda h: ("derivatives" in h.lower(), h))
    return hits[0] if hits else None


def _norm(ch):
    c = ch.lower().replace("eeg", "").strip(" -_")
    c = re.split(r"[-_ ]", c)[0]
    return ALIAS.get(c, c)


def load_raw(path):
    """19채널을 이름으로 골라 Mumtaz 순서로 정렬. 없거나 신호가 0인 채널은 보간. (raw, 보간한 채널 목록) 반환"""
    import mne
    raw = mne.io.read_raw_eeglab(path, preload=True, verbose="ERROR") if path.lower().endswith(".set") \
        else mne.io.read_raw(path, preload=True, verbose="ERROR")
    mont = mne.channels.make_standard_montage("standard_1005")
    canon = {n.lower(): n for n in mont.ch_names}
    rename, used = {}, set()
    for c in raw.ch_names:
        k = _norm(c)
        if k in canon and canon[k] not in used:
            rename[c] = canon[k]; used.add(canon[k])
    if "Cz" not in used:                                      # HydroCel 기준전극(VREF)은 Cz 위치
        for c in raw.ch_names:
            if c.upper() in ("VREF", "REF", "E65", "CZ_REF") and c not in rename:
                rename[c] = "Cz"; used.add("Cz"); break
    raw = raw.pick(list(rename)).rename_channels(rename)
    raw.set_channel_types({c: "eeg" for c in raw.ch_names}, verbose="ERROR")
    missing = [t for t in TARGET_19 if t not in raw.ch_names]
    if missing:                                               # 없는 채널은 0으로 추가한 뒤 보간
        info = mne.create_info(missing, raw.info["sfreq"], "eeg")
        raw.add_channels([mne.io.RawArray(np.zeros((len(missing), raw.n_times)), info, verbose="ERROR")],
                         force_update_info=True)
    raw.set_montage(mont, match_case=False, on_missing="ignore", verbose="ERROR")
    x = raw.get_data(picks=TARGET_19)
    flat = [t for t, s in zip(TARGET_19, x.std(1)) if s < 1e-9 and t not in missing]
    bads = missing + flat
    if bads:
        if len(raw.ch_names) - len(bads) < 20:
            raise RuntimeError(f"보간에 쓸 채널이 부족합니다: {raw.ch_names}")
        raw.info["bads"] = bads
        raw.interpolate_bads(reset_bads=True, origin="auto", verbose="ERROR")
    raw.pick(TARGET_19)
    return raw, bads


def preprocess(path):
    """reference 유지, 0.3–75 Hz, 50 Hz notch, 200 Hz, 5 s 비중첩, μV/100 -> [N,19,5,200]"""
    raw, interp = load_raw(path)
    fs_orig = raw.info["sfreq"]
    raw.filter(L_FREQ, H_FREQ, verbose="ERROR")
    raw.notch_filter(NOTCH, verbose="ERROR")
    raw.resample(FS_TARGET, verbose="ERROR")
    x = raw.get_data() * 1e6                                  # V -> μV
    n = x.shape[1] // (FS_TARGET * SEG_SEC)
    x = x[:, : n * FS_TARGET * SEG_SEC]
    segs = x.reshape(19, n, SEG_SEC, FS_TARGET).transpose(1, 0, 2, 3)
    segmax = np.abs(segs).reshape(n, -1).max(1)                # 구간별 최대 진폭(μV): 잡음 기준을 나중에 바꿀 수 있게 저장
    flag = segmax > FLAG_UV
    return (segs / AMP_DIV).astype(np.float32), flag, fs_orig, x.shape[1] / FS_TARGET, interp, segmax.astype(np.float32)


class _BraindecodeWrap:
    def __init__(self, m): self.m = m
    def parameters(self): return self.m.parameters()
    def __call__(self, x): return self.m(x.reshape(x.shape[0], x.shape[1], -1))


def build_cbramod(repo, weights):
    import torch
    if repo is None:
        from braindecode.models import CBraMod
        m = CBraMod.from_pretrained("braindecode/cbramod-pretrained", return_encoder_output=True).eval()
        m = m.to("cuda" if torch.cuda.is_available() else "cpu")
        for p in m.parameters(): p.requires_grad_(False)
        return _BraindecodeWrap(m)
    sys.path.insert(0, repo)
    from models.cbramod import CBraMod
    m = CBraMod(in_dim=200, out_dim=200, d_model=200, dim_feedforward=800, seq_len=30, n_layer=12, nhead=8)
    m.load_state_dict(torch.load(weights, map_location="cpu"))
    m.proj_out = torch.nn.Identity()
    m = m.to("cuda" if torch.cuda.is_available() else "cpu").eval()
    for p in m.parameters(): p.requires_grad_(False)
    return m


def extract_features(model, segs, bs=64):
    import torch
    dev = next(model.parameters()).device
    out = []
    with torch.no_grad():
        for i in range(0, len(segs), bs):
            z = model(torch.from_numpy(segs[i:i + bs]).to(dev))
            out.append(z.mean(dim=2).reshape(z.shape[0], -1).cpu().numpy())
    return np.concatenate(out).astype(np.float32)


# ----------------------------------------------------------------------------- inspect
def run_inspect(a):
    d = load_participants(a.participants, a.label_col, a.task)
    os.makedirs(a.out, exist_ok=True)
    print("라벨 열:", d.attrs["label_col"], "| 나이 열:", d.attrs["age_col"], "| 성별 열:", d.attrs["sex_col"])
    print(describe(d).to_string(index=False), "\n")
    rows = []
    skip = {"participant_id", "subject", "label", d.attrs["label_col"]}
    for c in d.columns:
        if c in skip:
            continue
        hc, pdg = d[d.label == 0][c], d[d.label == 1][c]
        num = pd.to_numeric(d[c], errors="coerce")
        r = {"column": c, f"{N0} 값 있음 %": round(hc.notna().mean() * 100), f"{N1} 값 있음 %": round(pdg.notna().mean() * 100),
             "n_unique": d[c].nunique(), "예시": ", ".join(map(str, d[c].dropna().unique()[:4]))}
        if num.notna().sum() > 0.5 * d[c].notna().sum():
            r.update({f"{N0} 평균": round(num[d.label == 0].mean(), 2), f"{N1} 평균": round(num[d.label == 1].mean(), 2)})
        rows.append(r)
    tab = pd.DataFrame(rows)
    tab.to_csv(os.path.join(a.out, "participants_columns.csv"), index=False)
    pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 40)
    print(tab.to_string(index=False), "\n")
    cfg = suggest_config(d, tab)
    with open(os.path.join(a.out, "clinical_config_suggested.json"), "w") as f:
        json.dump(cfg, f, indent=2, ensure_ascii=False)
    print("임상 변수 분류 제안:", json.dumps(cfg, ensure_ascii=False, indent=1))
    if a.eeg_root:
        f = find_file(a.eeg_root, d.subject.iloc[0])
        print("\n첫 EEG 파일:", f)
        if f:
            import mne
            r0 = mne.io.read_raw_eeglab(f, preload=True, verbose="ERROR") if f.endswith(".set") else mne.io.read_raw(f, preload=True, verbose="ERROR")
            print(f"채널 {len(r0.ch_names)}개 | fs {r0.info['sfreq']} Hz | 길이 {r0.times[-1]:.1f} s")
            print("채널 이름:", r0.ch_names)
            raw, interp = load_raw(f)
            x = raw.get_data() * 1e6
            print("19채널 매핑 결과:", raw.ch_names, "| 보간한 채널:", interp or "없음")
            print("채널별 표준편차(μV):", dict(zip(raw.ch_names, np.round(x.std(1), 1))))


def suggest_config(d, tab):
    """규칙: HC 값이 거의 없고 PD에만 있으면 누출 / 증상·질병 척도 이름이면 증상 근접 / 나머지는 맥락"""
    ctx, sym, leak = [], [], []
    for _, r in tab.iterrows():
        c = r["column"]
        if r["n_unique"] <= 1 or (r[f"{N0} 값 있음 %"] == 0 and r[f"{N1} 값 있음 %"] == 0):
            continue
        if r["n_unique"] > 2 and pd.to_numeric(d[c], errors="coerce").notna().sum() < 0.5 * d[c].notna().sum():
            continue                                          # 범주가 많은 문자열 열은 제외
        if r[f"{N0} 값 있음 %"] < 20 and r[f"{N1} 값 있음 %"] >= 50:
            leak.append(c)
        elif any(k in c.lower() for k in SYMPTOM_KEYS):
            sym.append(c)
        else:
            ctx.append(c)
    return {"context": ctx, "symptom": sym, "leak": leak}


# ----------------------------------------------------------------------------- extract
def run_extract(a):
    d = select_subjects(load_participants(a.participants, a.label_col, a.task), "full")
    if a.limit:
        d = pd.concat([g.head(a.limit) for _, g in d.groupby("label")])
    fdir = a.feat_dir or os.path.join(a.out, "features"); os.makedirs(fdir, exist_ok=True)
    model = build_cbramod(a.cbramod_repo, a.weights)
    print("device:", next(model.parameters()).device)
    log, t0 = [], time.time()
    old = os.path.join(a.out, "extract_log.csv")
    prev = pd.read_csv(old).set_index(["subject", "cond"]) if os.path.exists(old) else None
    for _, r in d.iterrows():
        cond = CONDITIONS[0]
        fp = os.path.join(fdir, f"{r.subject}_{cond}.npz")
        rec = {"subject": r.subject, "label": int(r.label), "cond": cond}
        if os.path.exists(fp):
            z = np.load(fp); rec.update(status="cached", n_seg=len(z["feat"]), n_flag=int(z["flag"].sum()))
            if prev is not None and (r.subject, cond) in prev.index:
                rec["interp"] = prev.loc[(r.subject, cond)].get("interp", "")
            log.append(rec); continue
        path = find_file(a.eeg_root, r.subject)
        if path is None:
            rec.update(status="no_file"); log.append(rec); continue
        try:
            segs, flag, fs, dur, interp, segmax = preprocess(path)
            if len(segs) == 0:
                rec.update(status="too_short"); log.append(rec); continue
            np.savez_compressed(fp, feat=extract_features(model, segs), flag=flag, segmax=segmax)
            rec.update(status="ok", file=path, fs_orig=fs, dur_s=round(dur, 1), n_seg=len(segs),
                       n_flag=int(flag.sum()), interp="+".join(interp),
                       segmax_median=round(float(np.median(segmax)), 1), over200=int((segmax > 200).sum()), over300=int((segmax > 300).sum()))
        except Exception as e:
            rec.update(status=f"error: {e}", file=path)
        log.append(rec)
        print(rec, flush=True)
    log = pd.DataFrame(log)
    log.to_csv(old, index=False)
    print(log.status.str.split(":").str[0].value_counts().to_string())
    print(f"done in {(time.time() - t0) / 60:.1f} min")


# ----------------------------------------------------------------------------- 실험 1: 교차검증
def metrics(y, p, thr=0.5):
    from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score, f1_score, confusion_matrix
    yh = (np.asarray(p) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0, 1]).ravel()
    return {"AUROC": roc_auc_score(y, p) if len(set(y)) > 1 else np.nan,
            "BalAcc": balanced_accuracy_score(y, yh), "Acc": accuracy_score(y, yh),
            "Sens": tp / (tp + fn) if tp + fn else np.nan, "Spec": tn / (tn + fp) if tn + fp else np.nan,
            "F1": f1_score(y, yh, zero_division=0), "TN": tn, "FP": fp, "FN": fn, "TP": tp}


def load_feature_table(fdir, subjects, with_flag=False, flag_uv=None):
    X, rows = [], []
    for sid in subjects:
        for cond in CONDITIONS:
            fp = os.path.join(fdir, f"{sid}_{cond}.npz")
            if os.path.exists(fp):
                z = np.load(fp); X.append(z["feat"])
                fl = (z["segmax"] > flag_uv) if (flag_uv and "segmax" in z.files) else z["flag"]   # --flag-uv 로 기준 변경
                rows += [(sid, cond, bool(f)) for f in fl]
    meta = pd.DataFrame(rows, columns=["subject", "cond", "flag"])
    return np.concatenate(X), (meta if with_flag else meta[["subject", "cond"]])


def subject_scores(meta, prob):
    m = meta.assign(p=prob)
    return m.groupby(["subject", "cond"]).p.mean().groupby("subject").mean()


def _fit_eeg(X, y):
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    sc = StandardScaler().fit(X)
    return sc, LogisticRegression(C=C_REG, max_iter=5000).fit(sc.transform(X), y)


def cohort_data(a, cohort, X=None, meta=None, with_flag=False):
    d = select_subjects(load_participants(a.participants, a.label_col, a.task), cohort)
    if X is None:
        X, meta = load_feature_table(a.feat_dir or os.path.join(a.out, "features"), d.subject, with_flag, getattr(a, "flag_uv", None))
    attrs = d.attrs
    d = d[d.subject.isin(meta.subject.unique())].reset_index(drop=True); d.attrs = attrs
    keep = meta.subject.isin(set(d.subject)).values
    return d, X[keep], meta[keep].reset_index(drop=True)


def run_cv(a, X=None, meta=None, seed=None, tag=None, quiet=False):
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import average_precision_score
    seed = SEED if seed is None else seed
    d, X, meta = cohort_data(a, a.cohort, X, meta)
    lab = dict(zip(d.subject, d.label)); y_seg = meta.subject.map(lab).values
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    oof = np.full(len(meta), np.nan); fold_rows = []; subj_fold = {}
    for k, (tr, te) in enumerate(skf.split(d.subject, d.label), 1):
        tr_s, te_s = set(d.subject[tr]), set(d.subject[te])
        itr, ite = meta.subject.isin(tr_s).values, meta.subject.isin(te_s).values
        sc, clf = _fit_eeg(X[itr], y_seg[itr])
        oof[ite] = clf.predict_proba(sc.transform(X[ite]))[:, 1]
        s = subject_scores(meta[ite], oof[ite])
        fold_rows.append({"fold": k, "n_train": len(tr_s), "n_test": len(te_s), **metrics(s.index.map(lab).values, s.values)})
        subj_fold.update({sid: k for sid in te_s})
    s = subject_scores(meta, oof); y = s.index.map(lab).values
    res = {"cohort": a.cohort, "seed": seed, "n_subjects": len(s), "n_HC": int((y == 0).sum()), "n_PD": int((y == 1).sum()),
           "n_recordings": int(meta.groupby(["subject", "cond"]).ngroups), "n_segments": len(meta),
           "subject_level": metrics(y, s.values), "segment_level": metrics(y_seg, oof)}
    res["subject_level"]["AUPRC"] = average_precision_score(y, s.values)
    folds = pd.DataFrame(fold_rows)
    od = os.path.join(a.out, tag or f"cv_{a.cohort}"); os.makedirs(od, exist_ok=True)
    folds.to_csv(os.path.join(od, "folds.csv"), index=False)
    pd.DataFrame({"subject": s.index, "label": y, "p_pd": s.values, "fold": s.index.map(subj_fold)}).to_csv(
        os.path.join(od, "subject_oof.csv"), index=False)
    describe(d).to_csv(os.path.join(od, "demographics.csv"), index=False)
    with open(os.path.join(od, "summary.json"), "w") as f:
        json.dump(res, f, indent=2, default=float)
    if not quiet:
        sl = res["subject_level"]
        print(f"[{a.cohort}] subjects={res['n_subjects']} ({N0} {res['n_HC']}, {N1} {res['n_PD']}), segments={res['n_segments']}")
        print(f"  subject OOF: AUROC {sl['AUROC']:.4f} | BalAcc {sl['BalAcc']*100:.2f}% | Acc {sl['Acc']*100:.2f}% | "
              f"Sens {sl['Sens']*100:.2f}% | Spec {sl['Spec']*100:.2f}% | F1 {sl['F1']*100:.2f}% | AUPRC {sl['AUPRC']:.4f}")
        print(f"  confusion: TN={sl['TN']} FP={sl['FP']} FN={sl['FN']} TP={sl['TP']}")
        print(f"  fold AUROC {folds.AUROC.mean():.4f} ± {folds.AUROC.std():.4f} | "
              f"fold BalAcc {folds.BalAcc.mean()*100:.2f} ± {folds.BalAcc.std()*100:.2f}%p")
        print(f"  segment OOF: AUROC {res['segment_level']['AUROC']:.4f} | BalAcc {res['segment_level']['BalAcc']*100:.2f}%")
    return res


def run_seeds(a):
    rows = []
    for s in range(a.n_seeds):
        r = run_cv(a, seed=s, tag=f"seeds_{a.cohort}/s{s}", quiet=True)["subject_level"]
        rows.append({"seed": s, **{k: round(r[k], 4) for k in ["AUROC", "BalAcc", "Sens", "Spec"]}})
    t = pd.DataFrame(rows); t.to_csv(os.path.join(a.out, f"seeds_{a.cohort}.csv"), index=False)
    print(f"[{a.cohort}] 분할 seed {a.n_seeds}개 반복")
    print(t.drop(columns="seed").agg(["mean", "std", "min", "max"]).round(4).to_string())


# ----------------------------------------------------------------------------- 실험 2: 잡음 점검
def run_noise(a):
    from scipy.stats import mannwhitneyu, spearmanr
    from sklearn.metrics import roc_auc_score
    d, X, meta = cohort_data(a, a.cohort, with_flag=True)
    thr = a.flag_uv or FLAG_UV
    base = f"noise_{a.cohort}" + (f"_uv{int(a.flag_uv)}" if a.flag_uv else "")
    od = os.path.join(a.out, base); os.makedirs(od, exist_ok=True)
    print(f"잡음 기준: 5초 구간 최대 진폭 > {thr:g} μV")
    lab = dict(zip(d.subject, d.label))
    # (1) 전체 vs 잡음 구간 제외
    print("##### 전체 구간"); r_all = run_cv(a, X, meta[["subject", "cond"]], tag=f"{base}/all")
    k = ~meta.flag.values
    print(f"\n##### {thr:g} μV 초과 구간 제외"); r_nf = run_cv(a, X[k], meta[k][["subject", "cond"]].reset_index(drop=True),
                                                  tag=f"{base}/noflag")
    t1 = pd.DataFrame([{"setting": n, "n_subj": r["n_subjects"], "n_seg": r["n_segments"],
                        **{m: round(r["subject_level"][m], 4) for m in ["AUROC", "BalAcc", "Sens", "Spec"]}}
                       for n, r in [("all", r_all), ("noflag", r_nf)]])
    # (2) 그룹별 잡음 비율 + 잡음만으로 분류
    ratio = meta.groupby("subject").flag.mean(); y = ratio.index.map(lab).values
    hc, pdg = ratio[y == 0], ratio[y == 1]
    t2 = pd.DataFrame([{f"{N0}_%": round(hc.mean() * 100, 1), f"{N1}_%": round(pdg.mean() * 100, 1),
                        "많은 쪽": N1 if pdg.mean() > hc.mean() else N0,
                        "MannWhitney_p": round(mannwhitneyu(hc, pdg).pvalue, 4),
                        "noise_only_AUROC": round(roc_auc_score(y, ratio.values), 4)}])
    # (3) 같은 그룹 안에서 라벨 1({N1}) 확률과 잡음 비율의 상관
    oof = pd.read_csv(os.path.join(a.out, f"{base}/all/subject_oof.csv")).set_index("subject")
    oof["noise"] = ratio
    t3 = pd.DataFrame([{"group": N0 if l == 0 else N1, "n": len(g),
                        "spearman_r": round(spearmanr(g.p_pd, g.noise).correlation, 3),
                        "p": round(spearmanr(g.p_pd, g.noise).pvalue, 4)} for l, g in oof.groupby("label")])
    for n, t in [("eval", t1), ("noise_by_group", t2), ("pred_vs_noise", t3)]:
        t.to_csv(os.path.join(od, f"{n}.csv"), index=False)
    print("\n========== (1) 전체 vs 잡음 구간 제외"); print(t1.to_string(index=False))
    print(f"\n========== (2) 그룹별 고진폭 구간 비율 ({thr:g} μV 초과, 피험자별 비율의 평균)"); print(t2.to_string(index=False))
    print(f"\n========== (3) 같은 그룹 안에서 라벨 1({N1}) 확률과 잡음 비율의 상관"); print(t3.to_string(index=False))


# ----------------------------------------------------------------------------- 실험 3·4 공통: 임상 변수, EEG 점수
def load_config(a, d):
    p = a.clinical_config or os.path.join(a.out, "clinical_config.json")
    if not os.path.exists(p):
        p = os.path.join(a.out, "clinical_config_suggested.json")
    with open(p) as f:
        cfg = json.load(f)
    for k in ["context", "symptom", "leak"]:
        cfg[k] = [c for c in cfg.get(k, []) if c in d.columns]
        if d.attrs.get("task") == "cog":                      # 라벨을 만든 MoCA는 입력에서 제외
            cfg[k] = [c for c in cfg[k] if c.lower() != "moca"]
    return cfg


def encode_clinical(d, cfg):
    """맥락·증상: 숫자로 변환(2범주 문자열은 0/1). 누출: 값이 있는지(has_X, 0/1). 결측은 NaN으로 두고 학습 fold 중앙값으로 대체"""
    C = pd.DataFrame(index=d.subject)
    names = {"context": [], "symptom": [], "leak": []}
    for grp in ["context", "symptom"]:
        for c in cfg[grp]:
            v = d.set_index("subject")[c]
            num = pd.to_numeric(v, errors="coerce")
            if num.notna().sum() < 0.5 * v.notna().sum():     # 문자열 -> 2범주만 0/1
                cats = sorted(v.dropna().astype(str).unique())
                if len(cats) != 2:
                    print(f"  [제외] {c}: 범주 {len(cats)}개"); continue
                num = v.astype(str).map({cats[0]: 0.0, cats[1]: 1.0}).where(v.notna())
                print(f"  [변환] {c}: {cats[0]}=0, {cats[1]}=1")
            C[c] = num.values; names[grp].append(c)
    for c in cfg["leak"]:
        C[f"has_{c}"] = d.set_index("subject")[c].notna().astype(float).values
        names["leak"].append(f"has_{c}")
    return C, names


def _logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def eeg_scores(a, d, X, meta):
    """outer 5-fold별 EEG 점수 (logit). 평가 fold = 학습 fold 전체로 학습한 EEG 모델 예측(실험 1과 동일),
    학습 fold = 학습 fold 안의 내부 5-fold out-of-fold 예측 (과적합된 점수로 결합 모델을 학습하는 누수 방지)"""
    from sklearn.model_selection import StratifiedKFold
    cache = os.path.join(a.out, f"clinical_{a.cohort}", "eeg_scores.csv")
    if os.path.exists(cache):
        return pd.read_csv(cache)
    os.makedirs(os.path.dirname(cache), exist_ok=True)
    lab = dict(zip(d.subject, d.label)); y_seg = meta.subject.map(lab).values
    rows = []
    outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    for k, (tr, te) in enumerate(outer.split(d.subject, d.label), 1):
        t0 = time.time()
        trs, tes = d.subject.values[tr], d.subject.values[te]
        itr, ite = meta.subject.isin(set(trs)).values, meta.subject.isin(set(tes)).values
        sc, clf = _fit_eeg(X[itr], y_seg[itr])
        s = subject_scores(meta[ite], clf.predict_proba(sc.transform(X[ite]))[:, 1])
        rows += [{"fold": k, "role": "test", "subject": i, "eeg": _logit(v)} for i, v in s.items()]
        inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        ytr = d.label.values[tr]
        for itr2, ite2 in inner.split(trs, ytr):
            a_s, b_s = set(trs[itr2]), set(trs[ite2])
            ia, ib = meta.subject.isin(a_s).values, meta.subject.isin(b_s).values
            sc2, clf2 = _fit_eeg(X[ia], y_seg[ia])
            s2 = subject_scores(meta[ib], clf2.predict_proba(sc2.transform(X[ib]))[:, 1])
            rows += [{"fold": k, "role": "train", "subject": i, "eeg": _logit(v)} for i, v in s2.items()]
        print(f"  EEG 점수 fold {k}/5 ({time.time() - t0:.0f} s)", flush=True)
    out = pd.DataFrame(rows); out.to_csv(cache, index=False)
    return out


def _fit_fusion(Xtr, ytr):
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    med = np.nanmedian(Xtr, axis=0); med = np.where(np.isnan(med), 0.0, med)
    Xtr = np.where(np.isnan(Xtr), med, Xtr)
    sc = StandardScaler().fit(Xtr)
    clf = LogisticRegression(C=C_FUSION, max_iter=5000).fit(sc.transform(Xtr), ytr)
    def predict(Xte):
        return clf.predict_proba(sc.transform(np.where(np.isnan(Xte), med, Xte)))[:, 1]
    return predict, clf.coef_[0]


def _frame(scores, C, lab, fold, role):
    s = scores[(scores.fold == fold) & (scores.role == role)].set_index("subject")
    f = C.loc[s.index].copy(); f["eeg"] = s.eeg; f["y"] = [lab[i] for i in s.index]
    return f


def _cv_combo(frames, cols, use_eeg):
    """outer fold별로 학습 fold(EEG 점수 = 내부 OOF)에서 학습, 평가 fold 예측. 전체 OOF 예측과 EEG 비중 반환"""
    feats = (["eeg"] if use_eeg else []) + list(cols)
    preds, shares = [], []
    for tr, te in frames:
        f, coef = _fit_fusion(tr[feats].values, tr.y.values)
        preds.append(pd.Series(f(te[feats].values), index=te.index))
        if use_eeg:
            shares.append(abs(coef[0]) / max(np.abs(coef).sum(), 1e-12))
    return pd.concat(preds), (float(np.mean(shares)) if shares else np.nan)


def _auc(y, p):
    from sklearn.metrics import roc_auc_score
    return roc_auc_score(y, p) if len(set(y)) > 1 else np.nan


def _inner_best(tr, combos, use_eeg=True):
    """학습 fold 안에서만 내부 5-fold로 조합 선택 (중첩 선택)"""
    from sklearn.model_selection import StratifiedKFold
    inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    splits = list(inner.split(tr.index, tr.y))
    best, best_auc = None, -1
    for cols in combos:
        feats = (["eeg"] if use_eeg else []) + list(cols)
        p = np.zeros(len(tr))
        for a_i, b_i in splits:
            f, _ = _fit_fusion(tr.iloc[a_i][feats].values, tr.iloc[a_i].y.values)
            p[b_i] = f(tr.iloc[b_i][feats].values)
        auc = _auc(tr.y.values, p)
        if auc > best_auc:
            best, best_auc = cols, auc
    return best


# ----------------------------------------------------------------------------- 실험 3: 임상정보 조합
def run_clinical(a):
    d, X, meta = cohort_data(a, a.cohort)
    cfg = load_config(a, d)
    print("임상 변수 설정:", cfg)
    C, names = encode_clinical(d, cfg)
    lab = dict(zip(d.subject, d.label))
    od = os.path.join(a.out, f"clinical_{a.cohort}"); os.makedirs(od, exist_ok=True)
    scores = eeg_scores(a, d, X, meta)
    frames = [(_frame(scores, C, lab, k, "train"), _frame(scores, C, lab, k, "test")) for k in range(1, 6)]
    y_all = pd.Series(lab)

    def evaluate(cols, use_eeg):
        p, share = _cv_combo(frames, cols, use_eeg)
        y = y_all.loc[p.index].values
        m = metrics(y, p.values)
        return {"AUROC": round(m["AUROC"], 4), "BalAcc": round(m["BalAcc"], 4), "EEG_share": round(share, 3)}

    # 그룹별 값 보유율
    rows = []
    for grp in ["context", "symptom", "leak"]:
        for c in cfg[grp]:
            v = d.set_index("subject")[c]
            rows.append({"group": grp, "variable": c, f"{N0} 값 있음 %": round(v[d.label.values == 0].notna().mean() * 100),
                         f"{N1} 값 있음 %": round(v[d.label.values == 1].notna().mean() * 100)})
    avail = pd.DataFrame(rows); avail.to_csv(os.path.join(od, "variable_availability.csv"), index=False)

    base = evaluate([], True)
    print(f"\nEEG-only (같은 방식으로 재계산): AUROC {base['AUROC']} | BalAcc {base['BalAcc']}")
    pool = names["context"] + names["symptom"]
    if len(pool) > MAX_COMBO_VARS:
        print(f"  변수 {len(pool)}개 중 앞 {MAX_COMBO_VARS}개만 조합에 사용 (MAX_COMBO_VARS)")
        pool = pool[:MAX_COMBO_VARS]
    combos = [c for r in range(1, len(pool) + 1) for c in itertools.combinations(pool, r)]
    print(f"조합 {len(combos)}개 평가 중...", flush=True)
    rows = []
    for cols in combos:
        clin, fus = evaluate(cols, False), evaluate(cols, True)
        rows.append({"variables": "+".join(cols), "n_vars": len(cols),
                     "has_symptom": any(c in names["symptom"] for c in cols),
                     "clinical_AUROC": clin["AUROC"], "fusion_AUROC": fus["AUROC"], "fusion_BalAcc": fus["BalAcc"],
                     "dAUROC_vs_EEG": round(fus["AUROC"] - base["AUROC"], 4), "EEG_share": fus["EEG_share"]})
    allc = pd.DataFrame(rows).sort_values("fusion_AUROC", ascending=False)
    allc.to_csv(os.path.join(od, "all_combinations.csv"), index=False)
    single = allc[allc.n_vars == 1].sort_values("dAUROC_vs_EEG", ascending=False)
    single.to_csv(os.path.join(od, "single_variables.csv"), index=False)

    # 중첩 선택
    nested = []
    pools = [("맥락만", [c for c in combos if not any(v in names["symptom"] for v in c)])]
    if names["symptom"]:
        pools.append(("맥락 + 증상 근접", combos))
    for name, cands in pools:
        if not cands:
            continue
        preds, chosen = [], []
        for tr, te in frames:
            best = _inner_best(tr, cands)
            feats = ["eeg"] + list(best)
            f, _ = _fit_fusion(tr[feats].values, tr.y.values)
            preds.append(pd.Series(f(te[feats].values), index=te.index)); chosen.append("+".join(best))
        p = pd.concat(preds); m = metrics(y_all.loc[p.index].values, p.values)
        nested.append({"pool": name, "AUROC": round(m["AUROC"], 4), "BalAcc": round(m["BalAcc"], 4),
                       "dAUROC_vs_EEG": round(m["AUROC"] - base["AUROC"], 4), "fold별 선택 조합": " · ".join(chosen)})
    nested = pd.DataFrame(nested); nested.to_csv(os.path.join(od, "nested_selection.csv"), index=False)

    # 누출 대조
    leak_rows = []
    for cols in [[c] for c in names["leak"]] + ([names["leak"]] if len(names["leak"]) > 1 else []):
        clin, fus = evaluate(cols, False), evaluate(cols, True)
        leak_rows.append({"setting": "EEG + " + "+".join(cols), "clinical_AUROC": clin["AUROC"],
                          "fusion_AUROC": fus["AUROC"], "fusion_BalAcc": fus["BalAcc"], "EEG_share": fus["EEG_share"]})
    leak = pd.DataFrame(leak_rows); leak.to_csv(os.path.join(od, "leakage_control.csv"), index=False)
    with open(os.path.join(od, "baseline.json"), "w") as f:
        json.dump({"EEG_only": base, "config": cfg, "encoded": names}, f, indent=2, ensure_ascii=False)

    pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 120)
    print("\n========== 변수별 그룹 값 보유율"); print(avail.to_string(index=False))
    print(f"\n========== (1) 조합 상위 10개 (고정 순위)  | 기준 EEG-only AUROC {base['AUROC']}, BalAcc {base['BalAcc']}")
    print(allc.head(10).to_string(index=False))
    print("\n========== (2) 단일 변수 효과"); print(single.to_string(index=False))
    print("\n========== (3) 중첩 선택 (선택 편향 없는 최상 조합 성능)"); print(nested.to_string(index=False))
    print("\n========== (4) 누출 대조"); print(leak.to_string(index=False) if len(leak) else "누출 변수 없음")


# ----------------------------------------------------------------------------- 실험 4: 정보 교체
def run_swap(a):
    d, X, meta = cohort_data(a, a.cohort)
    cfg = load_config(a, d)
    C, names = encode_clinical(d, cfg)
    lab = dict(zip(d.subject, d.label))
    od = os.path.join(a.out, f"clinical_{a.cohort}"); os.makedirs(od, exist_ok=True)
    scores = eeg_scores(a, d, X, meta)
    frames = [(_frame(scores, C, lab, k, "train"), _frame(scores, C, lab, k, "test")) for k in range(1, 6)]
    models = [("맥락 모델", names["context"])]
    if names["symptom"]:
        models.append(("맥락+증상 모델", names["context"] + names["symptom"]))
    if names["leak"]:
        models.append(("누출 모델", names["leak"]))
    models = [(n, c) for n, c in models if c]
    subj = np.array(list(lab)); ylab = np.array([lab[s] for s in subj])
    rng = np.random.default_rng(SEED)

    def donors(kind):
        out = {}
        for s, y in zip(subj, ylab):
            if kind == "random":   pool = subj[subj != s]
            elif kind == "same":   pool = subj[(ylab == y) & (subj != s)]
            else:                  pool = subj[ylab != y]
            out[s] = rng.choice(pool, N_SWAP, replace=True)
        return out
    donor = {k: donors(k) for k in ["random", "same", "opposite"]}

    res_rows, conflict_rows, per_subj = [], [], []
    for mname, cols in models:
        p_eeg, p_own, p_sw = {}, {}, {k: {} for k in donor}
        for tr, te in frames:
            f0, _ = _fit_fusion(tr[["eeg"]].values, tr.y.values)
            f1, _ = _fit_fusion(tr[["eeg"] + cols].values, tr.y.values)
            p0, p1 = f0(te[["eeg"]].values), f1(te[["eeg"] + cols].values)
            for i, s in enumerate(te.index):
                p_eeg[s], p_own[s] = p0[i], p1[i]
                for kind in donor:
                    clin = C.loc[donor[kind][s], cols].values             # 기증자 N_SWAP명의 임상정보
                    xx = np.column_stack([np.full(N_SWAP, te.loc[s, "eeg"]), clin])
                    p_sw[kind][s] = f1(xx)
        ss = list(p_own); y = np.array([lab[s] for s in ss])
        own = np.array([p_own[s] for s in ss]); eeg = np.array([p_eeg[s] for s in ss])
        res_rows.append({"model": mname, "condition": "1. 미제공 (EEG-only)", "AUROC": round(_auc(y, eeg), 4), "판정 변화 %": np.nan})
        res_rows.append({"model": mname, "condition": "2. 본인 정보", "AUROC": round(_auc(y, own), 4), "판정 변화 %": 0.0})
        for kind, label in [("random", "3. 무작위 다른 환자"), ("same", "4. 같은 그룹 다른 환자"), ("opposite", "5. 반대 그룹 환자")]:
            P = np.array([p_sw[kind][s] for s in ss])                     # [피험자, N_SWAP]
            aucs = [_auc(y, P[:, j]) for j in range(N_SWAP)]
            change = ((P >= 0.5) != (own[:, None] >= 0.5)).mean() * 100
            res_rows.append({"model": mname, "condition": label, "AUROC": round(np.mean(aucs), 4), "판정 변화 %": round(change, 1)})
            if kind == "opposite":
                dec = P >= 0.5; chart = (1 - y)[:, None]                  # 붙인 차트의 그룹
                eeg_dec = (eeg >= 0.5)[:, None]
                differs = np.broadcast_to(eeg_dec != chart, dec.shape)     # EEG-only 판정과 차트 방향이 다른 경우
                follow = ((dec == chart) & differs).sum() / max(differs.sum(), 1) * 100
                conflict_rows.append({"model": mname,
                                      f"{N0} 뇌파 + {N1} 차트 -> {N1} 판정 %": round(dec[y == 0].mean() * 100, 1),
                                      f"(기준) EEG-only {N0} -> {N1} 판정 %": round((eeg[y == 0] >= 0.5).mean() * 100, 1),
                                      f"{N1} 뇌파 + {N0} 차트 -> {N0} 판정 %": round((~dec[y == 1]).mean() * 100, 1),
                                      f"(기준) EEG-only {N1} -> {N0} 판정 %": round((eeg[y == 1] < 0.5).mean() * 100, 1),
                                      "차트를 따른 비율 %": round(follow, 1)})
                for i, s in enumerate(ss):
                    per_subj.append({"model": mname, "subject": s, "label": y[i], "p_eeg_only": eeg[i], "p_own": own[i],
                                     "p_opposite_mean": P[i].mean()})
    res = pd.DataFrame(res_rows); conf = pd.DataFrame(conflict_rows)
    res.to_csv(os.path.join(od, "swap_results.csv"), index=False)
    conf.to_csv(os.path.join(od, "swap_conflict.csv"), index=False)
    pd.DataFrame(per_subj).to_csv(os.path.join(od, "swap_per_subject.csv"), index=False)
    pd.set_option("display.width", 250)
    print("모델별 변수:", {n: c for n, c in models})
    print("\n========== 조건별 AUROC와 판정 변화 (판정 변화 = 본인 정보일 때와 PD/HC 판정이 달라진 비율, 기증자 20회 평균)")
    print(res.pivot(index="condition", columns="model", values=["AUROC", "판정 변화 %"]).to_string())
    print("\n========== 조건 5: EEG와 차트가 충돌할 때")
    print("  (차트를 따른 비율 = EEG-only 판정과 차트 방향이 다른 경우 중, 최종 판정이 차트 쪽으로 바뀐 비율)")
    print(conf.to_string(index=False))


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("step", choices=["inspect", "select", "extract", "cv", "seeds", "noise", "clinical", "swap"])
    ap.add_argument("--participants", required=True)
    ap.add_argument("--out", default="out_ad")
    ap.add_argument("--eeg-root"); ap.add_argument("--cbramod-repo"); ap.add_argument("--weights")
    ap.add_argument("--label-col", default=None)
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--cohort", default="full", choices=["full", "matched"])
    ap.add_argument("--n-seeds", type=int, default=10)
    ap.add_argument("--flag-uv", type=float, default=None, help="실험 2 잡음 기준(μV). 기본 100 (추출 때 저장한 flag)")
    ap.add_argument("--clinical-config", default=None)
    ap.add_argument("--task", default="ad_cn", choices=["ad_cn", "ad_ftd", "ftd_cn", "all"], help="ad_cn: AD vs CN / ad_ftd: AD vs FTD / ftd_cn: FTD vs CN / all: 추출용")
    ap.add_argument("--feat-dir", default=None, help="특징 폴더 (기본: OUT/features)")
    a = ap.parse_args()
    if a.step == "select":
        d0 = load_participants(a.participants, a.label_col, a.task); os.makedirs(a.out, exist_ok=True)
        for c in ["full", "matched"]:
            try:
                d = select_subjects(d0, c); d.to_csv(os.path.join(a.out, f"subjects_{c}.csv"), index=False)
                print(f"== {c}\n{describe(d).to_string(index=False)}\n")
            except RuntimeError as e:
                print(f"== {c}: {e}")
    else:
        {"inspect": run_inspect, "extract": run_extract, "cv": run_cv, "seeds": run_seeds, "noise": run_noise,
         "clinical": run_clinical, "swap": run_swap}[a.step](a)

In [ ]:
%%writefile /content/ad_extra.py
"""
AD-EEG 추가 실험 (연구 계획: EEG 근거를 보존하는 선택적 임상정보 활용)
ad_cbramod.py와 같은 폴더에 두고 실행. 실험 3·4와 같은 outer 5-fold(seed 42), 같은 EEG 점수 캐시를 사용.

  relate   : 실험 B. CBraMod 특징으로 임상 변수(UPDRS·MoCA·CTT·나이 등)를 예측 -> EEG-임상정보 관련성
  preserve : 실험 A. EEG 근거 보존 결합 시제품 3가지를 실험 4의 교체 조건으로 평가
             A1 변화 폭 제한   : 최종 logit = EEG logit + clip(임상 보정, ±δ)
             A2 교체 정보 학습 : 학습 때 다른 환자 차트를 붙인 사본(정답은 본인 것)을 ρ배 추가
             A3 일관성 가중치  : 최종 logit = EEG logit + w·(임상 보정),
                                 w = 차트 값이 '이 EEG로 예측한 값'과 가까울수록 1, 멀수록 0 (폭 τ)

사용 예
  python ad_extra.py relate   --participants P --out OUT --cohort full
  python ad_extra.py preserve --participants P --out OUT --cohort matched
"""
import argparse, json, os, sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import ad_cbramod as B

DELTAS = [0.25, 0.5, 1.0, 2.0]
RHOS = [0.5, 1.0, 2.0, 4.0]
TAUS = [0.1, 0.25, 0.5, 1.0, 2.0]          # 0.1·0.25 추가 (더 엄격한 일관성)
RIDGE_ALPHAS = np.logspace(0, 5, 11)


def _sig(x): return 1 / (1 + np.exp(-x))


def subject_features(X, meta, subjects):
    """구간 특징 평균 -> 피험자당 3,800차원"""
    m = pd.DataFrame(X, index=meta.subject.values)
    return m.groupby(level=0).mean().loc[subjects].values


def _outer_splits(d):
    from sklearn.model_selection import StratifiedKFold
    return list(StratifiedKFold(n_splits=5, shuffle=True, random_state=B.SEED).split(d.subject, d.label))


def _ridge_fit(F, y):
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import RidgeCV
    sc = StandardScaler().fit(F)
    r = RidgeCV(alphas=RIDGE_ALPHAS).fit(sc.transform(F), y)
    return lambda G: r.predict(sc.transform(G))


def _ridge_oof(F, y, n=5):
    from sklearn.model_selection import KFold
    p = np.zeros(len(y))
    for a, b in KFold(n_splits=n, shuffle=True, random_state=B.SEED).split(F):
        p[b] = _ridge_fit(F[a], y[a])(F[b])
    return p


def _setup(a):
    d, X, meta = B.cohort_data(a, a.cohort)
    cfg = B.load_config(a, d)
    C, names = B.encode_clinical(d, cfg)
    lab = dict(zip(d.subject, d.label))
    F = subject_features(X, meta, d.subject.values)
    return d, X, meta, C, names, lab, F


# ----------------------------------------------------------------------------- 실험 B: EEG -> 임상 변수
def run_relate(a):
    from scipy.stats import spearmanr
    d, X, meta, C, names, lab, F = _setup(a)
    od = os.path.join(a.out, f"extra_{a.cohort}"); os.makedirs(od, exist_ok=True)
    targets = [c for c in names["context"] + names["symptom"] if C[c].nunique() > 2]
    if a.task == "cog":                                        # 라벨의 연속값(MoCA)도 예측 대상으로 (입력에는 안 씀)
        mc = B._find_col(d, r"^moca$")
        C = C.copy(); C["moca(라벨 원천)"] = pd.to_numeric(d.set_index("subject")[mc], errors="coerce").values
        targets.append("moca(라벨 원천)")
    y_lab = d.label.values
    rows, preds = [], {"subject": d.subject.values, "label": y_lab}
    for v in targets:
        y = C[v].values
        ok = ~np.isnan(y)
        oof = np.full(len(y), np.nan)
        for tr, te in _outer_splits(d):
            tr_ok, te_ok = tr[ok[tr]], te[ok[te]]
            if len(tr_ok) < 10 or len(te_ok) == 0:
                continue
            oof[te_ok] = _ridge_fit(F[tr_ok], y[tr_ok])(F[te_ok])
        preds[f"pred_{v}"] = oof
        r = {"variable": v, "group": "context" if v in names["context"] else ("symptom" if v in names["symptom"] else "label"), "n": int(ok.sum())}
        for gname, g in [("전체", ok), (f"{B.N0} 안에서", ok & (y_lab == 0)), (f"{B.N1} 안에서", ok & (y_lab == 1))]:
            if g.sum() >= 8:
                s = spearmanr(oof[g], y[g])
                r[f"r_{gname}"] = round(s.correlation, 3); r[f"p_{gname}"] = round(s.pvalue, 4)
        res = y[ok] - oof[ok]
        r["R2"] = round(1 - (res ** 2).sum() / ((y[ok] - y[ok].mean()) ** 2).sum(), 3)
        rows.append(r)
    t = pd.DataFrame(rows); t.to_csv(os.path.join(od, "eeg_predicts_clinical.csv"), index=False)
    pd.DataFrame(preds).to_csv(os.path.join(od, "eeg_predicted_values.csv"), index=False)
    pd.set_option("display.width", 250)
    print(f"[{a.cohort}] CBraMod 피험자 평균 특징 -> 임상 변수 (Ridge, outer 5-fold OOF 예측과 실제 값의 Spearman 상관)")
    print(f"  '{B.N1} 안에서'·'{B.N0} 안에서' 상관이 그룹 차이를 뺀 EEG-임상정보 관련성")
    print(t.to_string(index=False))


# ----------------------------------------------------------------------------- 실험 A: EEG 근거 보존 결합
def run_preserve(a):
    d, X, meta, C, names, lab, F = _setup(a)
    od = os.path.join(a.out, f"extra_{a.cohort}"); os.makedirs(od, exist_ok=True)
    scores = B.eeg_scores(a, d, X, meta)
    frames = [(B._frame(scores, C, lab, k, "train"), B._frame(scores, C, lab, k, "test")) for k in range(1, 6)]
    splits = _outer_splits(d)
    pos = {s: i for i, s in enumerate(d.subject.values)}
    models = [(n, c) for n, c in [("맥락+증상 모델", names["context"] + names["symptom"]), ("누출 모델", names["leak"])] if c]
    subj = d.subject.values; ylab = d.label.values
    rng = np.random.default_rng(B.SEED)
    donor = {}
    for kind in ["random", "same", "opposite"]:                 # 실험 4와 같은 방식의 기증자
        donor[kind] = {}
        for s, y in zip(subj, ylab):
            pool = subj[subj != s] if kind == "random" else (subj[(ylab == y) & (subj != s)] if kind == "same" else subj[ylab != y])
            donor[kind][s] = rng.choice(pool, B.N_SWAP, replace=True)

    methods = [("단순 결합", None)] + [("A1 변화 폭 제한", dl) for dl in DELTAS] + \
              [("A2 교체 정보 학습", r) for r in RHOS] + [("A3 일관성 가중치", t) for t in TAUS] + \
              [("A2+A3 조합", (r, t)) for r in [2.0, 4.0] for t in [0.5, 1.0]]
    rows = []
    for mname, cols in models:
        # P[method][kind][subject] = 예측 확률 배열 (own은 길이 1, 교체는 N_SWAP)
        P = {m: {k: {} for k in ["eeg", "own", "random", "same", "opposite"]} for m in methods}
        for k, ((tr, te), (itr, ite)) in enumerate(zip(frames, splits), 1):
            f0, _ = B._fit_fusion(tr[["eeg"]].values, tr.y.values)
            f1, _ = B._fit_fusion(tr[["eeg"] + cols].values, tr.y.values)
            aug = {}
            rng2 = np.random.default_rng(B.SEED + k)
            for r in RHOS:                                         # A2: 다른 환자 차트를 붙인 사본, 정답은 본인 것
                n_copy = int(np.ceil(r)); reps = []
                for c in range(n_copy):
                    take = np.ones(len(tr), bool) if c < int(r) else rng2.random(len(tr)) < (r - int(r))
                    t2 = tr[take].copy()
                    dn = [rng2.choice(tr.index[tr.index != s]) for s in t2.index]
                    t2[cols] = tr.loc[dn, cols].values
                    reps.append(t2)
                big = pd.concat([tr] + reps)
                aug[r], _ = B._fit_fusion(big[["eeg"] + cols].values, big.y.values)
            # A3: 학습 fold에서 EEG -> 각 임상 변수 예측 모델 + 잔차 표준편차(내부 OOF)
            Ftr = F[itr]; gate = []
            for c in cols:
                y = C.loc[subj[itr], c].values.astype(float); ok = ~np.isnan(y)
                med = np.nanmedian(y) if ok.any() else 0.0
                if ok.sum() < 10 or np.nanstd(y) == 0:
                    gate.append(None); continue
                fit = _ridge_fit(Ftr[ok], y[ok])
                sd = np.std(y[ok] - _ridge_oof(Ftr[ok], y[ok])) + 1e-9
                gate.append((fit, sd, med))
            for s in te.index:
                e = te.loc[s, "eeg"]
                own = te.loc[s, cols].values.astype(float)[None, :]
                fself = F[pos[s]][None, :]
                xhat = [g[0](fself)[0] if g else None for g in gate]
                charts = {"own": own}
                for kind in ["random", "same", "opposite"]:
                    charts[kind] = C.loc[donor[kind][s], cols].values.astype(float)
                l_e = np.log(np.clip(f0(np.array([[e]])), 1e-9, 1 - 1e-9) / np.clip(1 - f0(np.array([[e]])), 1e-9, 1))[0]
                for m in methods:
                    P[m]["eeg"][s] = _sig(np.array([l_e]))
                for kind, Z in charts.items():
                    xx = np.column_stack([np.full(len(Z), e), Z])
                    pf = np.clip(f1(xx), 1e-9, 1 - 1e-9); l_f = np.log(pf / (1 - pf))
                    P[("단순 결합", None)][kind][s] = pf
                    for dl in DELTAS:
                        P[("A1 변화 폭 제한", dl)][kind][s] = _sig(l_e + np.clip(l_f - l_e, -dl, dl))
                    for r in RHOS:
                        P[("A2 교체 정보 학습", r)][kind][s] = aug[r](xx)
                    # 일관성: 차트 값이 이 EEG로 예측한 값에서 몇 표준편차 떨어져 있는가 (결측은 중앙값 대체)
                    zs = []
                    for j, g in enumerate(gate):
                        if g is None:
                            continue
                        v = np.where(np.isnan(Z[:, j]), g[2], Z[:, j])
                        zs.append((v - xhat[j]) / g[1])
                    zs = np.array(zs) if zs else np.zeros((1, len(Z)))
                    for t in TAUS:
                        w = np.exp(-(zs ** 2) / (2 * t ** 2)).mean(axis=0)
                        P[("A3 일관성 가중치", t)][kind][s] = _sig(l_e + w * (l_f - l_e))
                    for r in [2.0, 4.0]:                         # A2로 학습한 결합 모델 + A3 일관성 가중치
                        pa = np.clip(aug[r](xx), 1e-9, 1 - 1e-9); l_a = np.log(pa / (1 - pa))
                        for t in [0.5, 1.0]:
                            w = np.exp(-(zs ** 2) / (2 * t ** 2)).mean(axis=0)
                            P[("A2+A3 조합", (r, t))][kind][s] = _sig(l_e + w * (l_a - l_e))
        ss = list(P[methods[0]]["own"]); y = np.array([lab[s] for s in ss])
        eeg = np.array([P[methods[0]]["eeg"][s][0] for s in ss])
        auc_eeg = B._auc(y, eeg)
        rows.append({"model": mname, "method": "EEG-only", "param": "", "AUROC_본인": round(auc_eeg, 4),
                     "AUROC_무작위": round(auc_eeg, 4), "AUROC_같은그룹": round(auc_eeg, 4), "AUROC_반대그룹": round(auc_eeg, 4),
                     "반대: 판정 변화 %": 0.0, "반대: 차트를 따른 %": 0.0, f"{B.N1}뇌파+{B.N0}차트 -> {B.N0} %": round((eeg[y == 1] < 0.5).mean() * 100, 1)})
        for m in methods:
            own = np.array([P[m]["own"][s][0] for s in ss])
            r = {"model": mname, "method": m[0], "param": "" if m[1] is None else (f"ρ{m[1][0]:g}·τ{m[1][1]:g}" if isinstance(m[1], tuple) else m[1]), "AUROC_본인": round(B._auc(y, own), 4)}
            for kind, lbl in [("random", "무작위"), ("same", "같은그룹"), ("opposite", "반대그룹")]:
                M = np.array([P[m][kind][s] for s in ss])
                r[f"AUROC_{lbl}"] = round(np.mean([B._auc(y, M[:, j]) for j in range(M.shape[1])]), 4)
                if kind == "opposite":
                    dec = M >= 0.5; chart = (1 - y)[:, None]
                    differs = np.broadcast_to(((eeg >= 0.5)[:, None]) != chart, dec.shape)
                    r["반대: 판정 변화 %"] = round(((M >= 0.5) != (own[:, None] >= 0.5)).mean() * 100, 1)
                    r["반대: 차트를 따른 %"] = round(((dec == chart) & differs).sum() / max(differs.sum(), 1) * 100, 1)
                    r[f"{B.N1}뇌파+{B.N0}차트 -> {B.N0} %"] = round((~dec[y == 1]).mean() * 100, 1)
            rows.append(r)
    t = pd.DataFrame(rows); t.to_csv(os.path.join(od, "preserve_results.csv"), index=False)
    pd.set_option("display.width", 300); pd.set_option("display.max_rows", 200)
    print(f"[{a.cohort}] EEG 근거 보존 결합 시제품 — 교체 조건별 평가 (기증자 {B.N_SWAP}회 평균)")
    print("  좋은 모델: AUROC_본인은 EEG-only보다 높고, AUROC_반대그룹·무작위는 EEG-only 근처를 유지하며, 차트를 따른 % 가 낮음")
    for mname, g in t.groupby("model", sort=False):
        print(f"\n===== {mname}"); print(g.drop(columns="model").to_string(index=False))


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("step", choices=["relate", "preserve"])
    ap.add_argument("--participants", required=True)
    ap.add_argument("--out", default="out_ad")
    ap.add_argument("--label-col", default=None)
    ap.add_argument("--cohort", default="full", choices=["full", "matched"])
    ap.add_argument("--clinical-config", default=None)
    ap.add_argument("--task", default="ad_cn", choices=["ad_cn", "ad_ftd", "ftd_cn"])
    ap.add_argument("--feat-dir", default=None)
    a = ap.parse_args()
    {"relate": run_relate, "preserve": run_preserve}[a.step](a)

In [ ]:
%%writefile /content/tdbrain_fusion.py
"""
TDBRAIN MDD vs HC — 임상정보 결합·정보 교체·EEG 근거 보존 결합 (PD-EEG·AD-EEG와 같은 코드)

이미 뽑아 둔 TDBRAIN 특징(out/features/{TDBRAIN_ID}_{EO|EC}.npz)을 그대로 쓰고,
PD-EEG·AD-EEG에서 쓴 ad_cbramod.py(실험 3·4)와 ad_extra.py(추가 B·A)를 TDBRAIN 형식에 맞춰 호출한다.
같은 폴더에 ad_cbramod.py, ad_extra.py 가 있어야 한다.

단계
  inspect  : 엑셀 열 목록, 그룹별 값 보유율, 변수 자동 대응표(tdbrain_mapping.json) 생성
  cv       : EEG-only 재확인 (TDBRAIN 보고서 matched 0.929와 같은지)
  seeds    : 분할 seed 10개
  clinical : 실험 3 (임상정보 조합)
  swap     : 실험 4 (정보 교체)
  relate   : 추가 B (EEG -> 임상 변수 예측)
  preserve : 추가 A (EEG 근거 보존 결합)

사용 예
  python tdbrain_fusion.py inspect  --participants X.xlsx --out OUT
  python tdbrain_fusion.py preserve --participants X.xlsx --out OUT --feat-dir FEAT --cohort matched
"""
import argparse, json, os, re, sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import ad_cbramod as B
import ad_extra as E

SEED = 42
MISSING_CODES = [99, 999, -99, -999]

# 표준 이름 -> 엑셀 열 이름을 찾는 정규식 (대소문자 무시). inspect가 실제 열에 대응시켜 tdbrain_mapping.json에 저장
PATTERNS = {
    "context": {"age": r"^age$", "gender": r"^(gender|sex)$", "education": r"educ", "sleep": r"sleep",
                "coffee": r"coffee", "cigarette": r"cigar|smok", "beer": r"beer|alcohol", "meal": r"meal"},
    "symptom": {"well": r"^well"},
    "leak": {"BDI": r"bdi", "rTMS": r"rtms", "NEO": r"neo"},          # 값이 하나라도 있으면 has_X = 1
}


# ----------------------------------------------------------------------------- 참가자 (TDBRAIN 보고서와 같은 선정)
def _raw(path):
    d = pd.read_excel(path)
    d["ind"] = d["indication"].astype(str).str.strip().str.upper()
    d = d[d["ind"].isin(["HEALTHY", "MDD"])].copy()
    d = d.sort_values("sessID").drop_duplicates("TDBRAIN_ID", keep="first")
    d = d[d["age"].notna() & (d["age"] >= 18)]
    d["label"] = (d["ind"] == "MDD").astype(int)
    d["subject"] = d["TDBRAIN_ID"].astype(str)
    return d.reset_index(drop=True)


def _apply_mapping(d, mapping):
    """표준 이름 열 만들기. 맥락·증상: 숫자 변환 + 결측 코드(99 등) 제거. 누출: 대응 열 중 하나라도 값이 있으면 1, 없으면 NaN"""
    for grp in ["context", "symptom"]:
        for std, cols in mapping.get(grp, {}).items():
            if not cols:
                continue
            v = pd.to_numeric(d[cols[0]], errors="coerce")
            d[std] = v.where(~v.isin(MISSING_CODES))
    for std, cols in mapping.get("leak", {}).items():
        if cols:
            d[std] = d[cols].notna().any(axis=1).map({True: 1.0, False: np.nan})
    return d


def _mapping_path(a): return os.path.join(a.out, "tdbrain_mapping.json")


def tdb_load(path, label_col=None, task=None, _a=None):
    B.N0, B.N1 = "HC", "MDD"
    d = _raw(path)
    mp = _mapping_path(_a) if _a else None
    if mp and os.path.exists(mp):
        d = _apply_mapping(d, json.load(open(mp)))
    d.attrs["task"] = "tdbrain"; d.attrs["age_col"] = "age"; d.attrs["sex_col"] = "gender"
    return d


def tdb_select(d, cohort="matched", caliper=3.0):
    """TDBRAIN 보고서와 같은 matched: HC 1명당 같은 성별·나이 차 3세 이내 최근접 MDD 1명 (seed 42)"""
    attrs = d.attrs
    if cohort == "full":
        out = d.copy()
    elif cohort == "matched":
        hc = d[d.label == 0].sample(frac=1, random_state=SEED)
        pool = d[d.label == 1].copy()
        keep = []
        for _, r in hc.iterrows():
            cand = pool[pool.gender == r.gender]
            if cand.empty:
                continue
            diff = (cand.age - r.age).abs()
            if diff.min() > caliper:
                continue
            j = diff.idxmin()
            keep += [r.TDBRAIN_ID, pool.loc[j, "TDBRAIN_ID"]]
            pool = pool.drop(j)
        out = d[d.TDBRAIN_ID.isin(keep)]
    else:
        raise ValueError(cohort)
    out = out.reset_index(drop=True); out.attrs = attrs
    return out


def _patch(a):
    """ad_cbramod의 참가자 읽기·선정·조건을 TDBRAIN용으로 바꿔 끼움"""
    B.CONDITIONS = ["EO", "EC"]
    B.load_participants = lambda p, lc=None, task=None: tdb_load(p, lc, task, a)
    B.select_subjects = tdb_select
    B.N0, B.N1 = "HC", "MDD"


# ----------------------------------------------------------------------------- inspect
def run_inspect(a):
    os.makedirs(a.out, exist_ok=True)
    d = _raw(a.participants)
    print(f"대상 (성인, 첫 세션, HEALTHY/MDD): HC {(d.label == 0).sum()}, MDD {(d.label == 1).sum()}")
    rows = []
    for c in d.columns:
        if c in ["ind", "label", "subject"]:
            continue
        v = d[c]
        num = pd.to_numeric(v, errors="coerce")
        r = {"column": c, "HC 값 있음 %": round(v[d.label == 0].notna().mean() * 100),
             "MDD 값 있음 %": round(v[d.label == 1].notna().mean() * 100), "n_unique": v.nunique()}
        if num.notna().sum() > 0.5 * v.notna().sum() and v.notna().any():
            r.update({"min": num.min(), "max": num.max(), "99 개수": int(num.isin(MISSING_CODES).sum())})
        rows.append(r)
    t = pd.DataFrame(rows)
    pd.set_option("display.width", 250); pd.set_option("display.max_rows", 500)
    print("\n========== 열 목록과 그룹별 값 보유율 (full 대상)"); print(t.to_string(index=False))
    t.to_csv(os.path.join(a.out, "participants_columns.csv"), index=False)

    mapping = {g: {} for g in PATTERNS}
    for g, pats in PATTERNS.items():
        for std, pat in pats.items():
            cols = [c for c in d.columns if re.search(pat, str(c), re.I) and c not in ["ind", "label", "subject"]]
            if g != "leak":
                # 맥락·증상은 열 하나만. BDI처럼 pre/post가 있으면 pre 우선
                cols = sorted(cols, key=lambda c: (not re.search(r"pre", c, re.I), len(c)))[:1]
            mapping[g][std] = cols
    json.dump(mapping, open(_mapping_path(a), "w"), indent=2, ensure_ascii=False)
    cfg = {g: [std for std, cols in m.items() if cols] for g, m in mapping.items()}
    json.dump(cfg, open(os.path.join(a.out, "clinical_config.json"), "w"), indent=2)
    print("\n임상 변수 설정 (clinical_config.json):", cfg)
    print("\n========== 변수 자동 대응 (tdbrain_mapping.json 에 저장, 틀리면 다음 셀에서 고쳐서 저장)")
    for g, m in mapping.items():
        for std, cols in m.items():
            print(f"  [{g}] {std:10s} <- {cols if cols else '못 찾음'}")


# ----------------------------------------------------------------------------- main
if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("step", choices=["inspect", "cv", "seeds", "clinical", "swap", "relate", "preserve"])
    ap.add_argument("--participants", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--feat-dir", default=None)
    ap.add_argument("--cohort", default="matched", choices=["matched", "full"])
    ap.add_argument("--n-seeds", type=int, default=10)
    ap.add_argument("--clinical-config", default=None)
    a = ap.parse_args()
    a.label_col, a.task, a.flag_uv = None, "tdbrain", None
    if a.step == "inspect":
        run_inspect(a)
    else:
        _patch(a)
        {"cv": B.run_cv, "seeds": B.run_seeds, "clinical": B.run_clinical, "swap": B.run_swap,
         "relate": E.run_relate, "preserve": E.run_preserve}[a.step](a)

In [ ]:
FEAT = f'{OUT}/features'                  # 이미 뽑아 둔 특징
OUT_F = f'{WORK}/out_fusion'              # 새 결과 폴더 (기존 결과와 섞이지 않게)
os.makedirs(OUT_F, exist_ok=True)
print('엑셀:', os.path.exists(XLSX), '| 특징 파일 수:', len(os.listdir(FEAT)))
def run(step, extra=''):
    get_ipython().system(f'python /content/tdbrain_fusion.py {step} --participants "{XLSX}" --out "{OUT_F}" --feat-dir "{FEAT}" {extra}')

In [ ]:
run('inspect')

누출 변수 중 BDI는 TDBRAIN 보고서와 같게 `BDI_pre`만 씁니다.

In [ ]:
import json
mp =json.load(open(f'{OUT_F}/tdbrain_mapping.json'))
mp['leak']['BDI'] = ['BDI_pre']
json.dump(mp, open(f'{OUT_F}/tdbrain_mapping.json', 'w'), indent=2, ensure_ascii=False)
print(mp['leak']['BDI'], '| rTMS:', mp['leak']['rTMS'], '| NEO 문항 수:', len(mp['leak']['NEO']))

In [ ]:
run('cv', '--cohort matched')

In [ ]:
for s in ['clinical', 'swap', 'relate', 'preserve']:
    print(f'\n#################### {s} / matched'); run(s, '--cohort matched')

## 15. 보고서용 요약

In [ ]:
import json, pandas as pd
for c in ['matched', 'full']:
    s = json.load(open(f'{OUT}/cv_{c}/summary.json'))
    print(f'== {c}:', {k: s[k] for k in ['n_subjects','n_HC','n_MDD','n_recordings','n_segments']})
    display(pd.Series(s['subject_level']).round(4).to_frame('subject OOF').T)
    display(pd.read_csv(f'{OUT}/cv_{c}/folds.csv').round(3))
    display(pd.read_csv(f'{OUT}/cv_{c}/demographics.csv'))